# Pistachio Cultivar Classification (Siirt vs. Kirmizi) — Revised Analysis Pipeline (v5.1)

This notebook reproduces **every** table, figure and statistical test of the revised manuscript from a
single run. All numbers are produced by the code below; nothing is entered manually.

**Pipeline.** Frozen ImageNet CNN backbones (ResNet50, EfficientNetB0, MobileNetV2) → per-block
standardization → PCA (95 % explained variance) → classical classifier (SVM, RF, KNN).

**Evaluation protocol.**
* Stratified 70 / 15 / 15 train / validation / test split, repeated with 5 random seeds
  (42, 0, 123, 7, 2025). The dataset contains visually identical copies of some images; these are grouped
  (Section 3.1) and every split is made at the group level, so copies never occur in different subsets.
* Repeated stratified group 5-fold cross-validation (5 × 5 = 25 folds) on the development set
  (train + validation of seed 42). The seed-42 test set is never used in cross-validation,
  model selection, PCA fitting, hyperparameter tuning or fusion-weight computation.
* Scaler, PCA and classifier are always fitted on training data only.
* Model selection uses cross-validation / validation data only, never test data.

**Outputs** are written to `CONFIG['output_dir']`: `figures/`, `tables/`, `predictions/`, `splits/`,
`features/`, `checkpoints/`.

**Runtime** (RTX 3050 laptop GPU, approximate): feature extraction 3–5 min, classical experiments
30–50 min, end-to-end fine-tuning 45–90 min (3 CNNs × 5 seeds), explainability 10–15 min.
Fine-tuning, augmentation and SHAP can be switched off in `CONFIG`.

### Changes with respect to v4
0. Near-duplicate images are detected and grouped; all splits and CV folds are group-aware (v5.1).
1. One deterministic, sorted file list and one set of split indices are shared by **all** experiments
   (hybrid models, fusion, baselines, fine-tuning). Split indices and per-image predictions are saved.
2. All 18 combinations are evaluated with 5 seeds (hold-out) and 5 × 5 repeated CV; results are reported
   as mean ± SD (sample SD, ddof = 1) with 95 % confidence intervals.
3. Additional metrics: balanced accuracy, macro-F1, weighted F1, sensitivity, specificity, precision,
   Cohen's kappa, MCC and ROC-AUC (positive class = Siirt; SVM decision function on the test set).
4. Friedman / Nemenyi on CV folds was replaced by the Nadeau–Bengio corrected resampled t-test on
   repeated CV (folds are not independent), with Holm correction and effect sizes (difference in
   percentage points with 95 % CI). Complete pairwise comparisons are reported.
5. Multi-CNN fusion: explicit formula, reported temperature τ, τ-sensitivity analysis, alternative
   fusion strategies, and all fusion / subset decisions made on validation data only. Fusion is
   compared with the validation-selected single CNN on 25 CV folds and 5 seeds.
6. Learning curve with shuffled subsets, starting from 5 % of the training data.
7. Explainability: occlusion sensitivity on the complete deployed pipeline (CNN → PCA → SVM);
   Grad-CAM only on end-to-end fine-tuned CNNs (where class gradients exist); quantitative
   attribution analysis (fraction of attribution inside the pistachio mask) on the full test set;
   objective example selection; SHAP explainer and background set documented.
8. New baselines: end-to-end fine-tuned CNNs (all three architectures) under the same splits.
9. Augmentation experiment for the selected hybrid model is reported.
10. Computational cost: backbone vs. full parameter counts, GPU timing with synchronization,
    warm-up, batch sizes 1 and 32, and the cost of PCA + classifier.

## 1. Setup

In [ ]:
# Install missing packages if needed (run once).
# %pip install -q scikit-learn matplotlib seaborn pandas tqdm statsmodels scikit-image grad-cam shap opencv-python

In [ ]:
import os, sys, json, time, copy, random, platform, hashlib, warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.manifold import TSNE
from sklearn.model_selection import (train_test_split, StratifiedKFold, StratifiedGroupKFold,
                                     GridSearchCV, learning_curve)
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, matthews_corrcoef, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
                             classification_report)

from scipy import stats, ndimage
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu

warnings.filterwarnings('ignore')

CONFIG = {
    # --- paths -------------------------------------------------------------------------------
    'data_dir'   : r'./Pistachio_Image_Dataset/Pistachio_Image_Dataset'   # path to the Kaggle dataset (folders: Kirmizi_Pistachio, Siirt_Pistachio),
    'output_dir' : 'results_v5_1',
    # --- data / splits -----------------------------------------------------------------------
    'img_size'   : 224,
    'batch_size' : 32,
    'seeds'      : [42, 0, 123, 7, 2025],
    'main_seed'  : 42,
    'test_frac'  : 0.15,
    'val_frac'   : 0.15,
    'cv_splits'  : 5,
    'cv_repeats' : 5,
    'dup_threshold': 0.99,          # cosine similarity for near-duplicate grouping
    'dup_feature'  : 'MobileNetV2', # features used for near-duplicate detection
    # --- hybrid models -----------------------------------------------------------------------
    'pca_var'    : 0.95,
    'svm_C'      : 10,
    'svm_gamma'  : 'scale',
    'rf_n'       : 200,
    'knn_k'      : 5,
    # --- multi-CNN fusion --------------------------------------------------------------------
    'fusion_taus'   : [1.0, 0.1, 0.05, 0.02, 0.01],
    'inner_val_frac': 0.15,   # inner validation split inside each CV fold (fusion weights)
    # --- learning curve ----------------------------------------------------------------------
    'lc_sizes'   : [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00],
    # --- end-to-end fine-tuning --------------------------------------------------------------
    'run_finetune'   : True,
    'ft_models'      : ['MobileNetV2', 'EfficientNetB0', 'ResNet50'],
    'ft_seeds'       : None,     # None = all seeds in CONFIG['seeds']
    'ft_epochs'      : 15,
    'ft_patience'    : 5,
    'ft_batch_size'  : 32,       # reduce to 16 if the GPU runs out of memory
    'ft_lr_backbone' : 1e-4,
    'ft_lr_head'     : 1e-3,
    'ft_weight_decay': 1e-4,
    'force_retrain'  : False,    # False = reuse saved checkpoints if they exist
    # --- augmentation experiment -------------------------------------------------------------
    'run_augmentation': True,
    'aug_copies'      : 4,
    # --- explainability ----------------------------------------------------------------------
    'occlusion_patch' : 32,
    'occlusion_stride': 16,
    'n_examples'      : 4,       # examples per group (correct / misclassified) in XAI figures
    'run_shap'        : True,
    'shap_n_explain'  : 100,
    'shap_background' : 50,
    'shap_nsamples'   : 500,
    # --- misc --------------------------------------------------------------------------------
    'n_bootstrap'     : 2000,
    'timing_runs'     : 100,
    'pretrained'      : True,
    'force_recompute_features': False,
}

In [ ]:
OUT = CONFIG['output_dir']
DIRS = {k: os.path.join(OUT, k) for k in
        ['figures', 'tables', 'predictions', 'splits', 'features', 'checkpoints']}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = list(CONFIG['seeds'])
MAIN_SEED = CONFIG['main_seed']
FT_SEEDS = SEEDS if CONFIG['ft_seeds'] is None else list(CONFIG['ft_seeds'])


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(MAIN_SEED)


def save_fig(name):
    path = os.path.join(DIRS['figures'], name)
    plt.savefig(path, dpi=300, bbox_inches='tight')
    return path


def save_table(df, name, index=False):
    path = os.path.join(DIRS['tables'], name)
    df.to_csv(path, index=index, encoding='utf-8-sig')
    return path


ENV = {
    'python': sys.version.split()[0], 'platform': platform.platform(),
    'processor': platform.processor(), 'torch': torch.__version__,
    'torchvision': torchvision.__version__, 'sklearn': sklearn.__version__,
    'numpy': np.__version__, 'cuda': torch.version.cuda,
    'device': str(DEVICE),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_memory_GB': round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
                     if torch.cuda.is_available() else None,
}
with open(os.path.join(OUT, 'environment.json'), 'w') as f:
    json.dump({'environment': ENV, 'config': CONFIG}, f, indent=2)
for k, v in ENV.items():
    print(f'{k:>14}: {v}')

## 2. Dataset

Pistachio Image Dataset (Özkan et al., 2021; Kaggle). Files are listed in sorted order so that the
image order, and therefore every split, is identical on every machine. Label 0 = Kirmizi,
label 1 = Siirt (positive class for sensitivity, specificity and ROC-AUC).

In [ ]:
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')
DATA_DIR = CONFIG['data_dir']
assert os.path.isdir(DATA_DIR), f'Data directory not found: {DATA_DIR}'

CLASS_DIRS = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
CLASS_DISPLAY = {'Kirmizi_Antep': 'Kirmizi', 'Kirmizi_Pistachio': 'Kirmizi', 'Siirt_Pistachio': 'Siirt'}
CLASS_NAMES = [CLASS_DISPLAY.get(d, d) for d in CLASS_DIRS]
assert len(CLASS_NAMES) == 2, CLASS_DIRS
POS_LABEL = 1

paths, labels = [], []
for ci, d in enumerate(CLASS_DIRS):
    files = sorted(f for f in os.listdir(os.path.join(DATA_DIR, d)) if f.lower().endswith(IMG_EXT))
    paths += [os.path.join(DATA_DIR, d, f) for f in files]
    labels += [ci] * len(files)
PATHS = np.array(paths)
LABELS = np.array(labels)
N = len(PATHS)

# Original image sizes and exact duplicate check (MD5)
sizes, md5s = [], []
for p in PATHS:
    with open(p, 'rb') as fh:
        md5s.append(hashlib.md5(fh.read()).hexdigest())
    with Image.open(p) as im:
        sizes.append(im.size)
file_df = pd.DataFrame({'index': np.arange(N),
                        'file': [os.path.relpath(p, DATA_DIR) for p in PATHS],
                        'label': LABELS, 'class': [CLASS_NAMES[l] for l in LABELS],
                        'width': [s[0] for s in sizes], 'height': [s[1] for s in sizes], 'md5': md5s})
file_df.to_csv(os.path.join(DIRS['splits'], 'file_list.csv'), index=False)

n_dup_groups = int((file_df.groupby('md5').size() > 1).sum())
print(f'Classes: {CLASS_NAMES} (folders: {CLASS_DIRS})')
for ci, c in enumerate(CLASS_NAMES):
    print(f'  {c}: {(LABELS == ci).sum()} images ({(LABELS == ci).mean() * 100:.1f} %)')
print(f'Total: {N} images')
print('Original image sizes:', file_df.groupby(['width', 'height']).size().to_dict())
print(f'Exact duplicate files (identical MD5): {n_dup_groups} groups')
print(f'Majority-class rate: {np.bincount(LABELS).max() / N * 100:.2f} %')

### 2.1 Image cache, grayscale conversion and pistachio masks

Images are resized to 224 × 224 (bilinear) and kept in memory. Grayscale images are produced with the
ITU-R 601 luma transform (PIL `convert('L')`) and replicated to three channels so that they can be fed
to ImageNet networks; the same ImageNet normalization (μ = [0.485, 0.456, 0.406],
σ = [0.229, 0.224, 0.225]) is applied to both modalities.

A binary pistachio mask (Otsu threshold, largest connected component, hole filling) is computed for each
image. It is used only to quantify explanation maps (Section 20).

In [ ]:
IMG_SIZE = CONFIG['img_size']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
_MEAN_T = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
_STD_T = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)


def load_resized(path, size):
    with Image.open(path) as im:
        return np.asarray(im.convert('RGB').resize((size, size), Image.BILINEAR))


IMG_RGB = np.stack([load_resized(p, IMG_SIZE) for p in tqdm(PATHS, desc='Loading images')])
IMG_GRAY = np.stack([np.asarray(Image.fromarray(a).convert('L')) for a in IMG_RGB])


def get_images(indices, modality):
    """uint8 array (B, H, W, 3) for the requested modality."""
    if modality == 'RGB':
        return IMG_RGB[indices]
    return np.repeat(IMG_GRAY[indices][..., None], 3, axis=-1)


def arrays_to_tensor(arr):
    t = torch.from_numpy(np.ascontiguousarray(arr)).permute(0, 3, 1, 2).float().div_(255.0)
    return (t - _MEAN_T) / _STD_T


def pistachio_mask(gray):
    thr = threshold_otsu(gray)
    border = np.concatenate([gray[0], gray[-1], gray[:, 0], gray[:, -1]])
    fg = gray > thr if border.mean() < thr else gray < thr
    fg = ndimage.binary_closing(fg, iterations=2)
    lab, n = ndimage.label(fg)
    if n > 1:
        sizes_ = ndimage.sum(fg, lab, range(1, n + 1))
        fg = lab == (np.argmax(sizes_) + 1)
    return ndimage.binary_fill_holes(fg)


MASKS = np.stack([pistachio_mask(g) for g in IMG_GRAY])
print(f'Mask area: {MASKS.mean(axis=(1, 2)).mean() * 100:.1f} % ± {MASKS.mean(axis=(1, 2)).std(ddof=1) * 100:.1f} % of the image')

rng = np.random.default_rng(MAIN_SEED)
fig, axes = plt.subplots(3, 6, figsize=(16, 8.5))
for ci in range(2):
    pick = rng.choice(np.where(LABELS == ci)[0], 3, replace=False)
    for j, k in enumerate(pick):
        col = ci * 3 + j
        axes[0, col].imshow(IMG_RGB[k]); axes[0, col].set_title(f'{CLASS_NAMES[ci]} — RGB', fontsize=9)
        axes[1, col].imshow(IMG_GRAY[k], cmap='gray'); axes[1, col].set_title('Grayscale', fontsize=9)
        axes[2, col].imshow(IMG_RGB[k]); axes[2, col].contour(MASKS[k], colors='lime', linewidths=1)
        axes[2, col].set_title('Pistachio mask', fontsize=9)
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle('Representative images, grayscale conversion and segmentation masks', fontweight='bold')
plt.tight_layout(); save_fig('fig_samples_masks.png'); plt.show()

### 2.2 Morphological description of the two cultivars (image-based)
Pixel-level size and color statistics derived from the masks (the dataset contains no weight / mass
information). Area is given in pixels of the original image.

In [ ]:
morph = []
for k in range(N):
    m = MASKS[k]
    if m.sum() == 0:
        continue
    scale = (sizes[k][0] * sizes[k][1]) / (IMG_SIZE * IMG_SIZE)
    rgb = IMG_RGB[k][m].astype(float)
    ys, xs = np.where(m)
    cov = np.cov(np.vstack([xs, ys]))
    ev = np.sort(np.linalg.eigvalsh(cov))[::-1]
    morph.append({'class': CLASS_NAMES[LABELS[k]], 'area_px': m.sum() * scale,
                  'aspect_ratio': np.sqrt(ev[0] / max(ev[1], 1e-9)),
                  'mean_R': rgb[:, 0].mean(), 'mean_G': rgb[:, 1].mean(), 'mean_B': rgb[:, 2].mean()})
morph_df = pd.DataFrame(morph)
morph_sum = morph_df.groupby('class').agg(['mean', 'std']).round(2)
save_table(morph_sum, 'table_cultivar_morphology.csv', index=True)
print(morph_sum.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['area_px', 'aspect_ratio', 'mean_G']):
    sns.histplot(data=morph_df, x=col, hue='class', kde=True, ax=ax, stat='density', common_norm=False)
    ax.set_title(col)
plt.tight_layout(); save_fig('fig_cultivar_morphology.png'); plt.show()

## 3. Deep feature extraction (frozen ImageNet backbones)

The final classification layer of each network is replaced by an identity mapping; the output of the
global average pooling layer is used as the feature vector (ResNet50: 2048-D, EfficientNetB0 and
MobileNetV2: 1280-D). The backbones are not trained; features are therefore computed once for all
images and cached.

In [ ]:
CNN_NAMES = ['ResNet50', 'EfficientNetB0', 'MobileNetV2']
MODALITIES = ['RGB', 'Grayscale']
CLF_NAMES = ['SVM', 'RF', 'KNN']


def build_backbone(name, pretrained=True, to_device=True):
    """Returns (backbone_without_head, grad_cam_target_layer, feature_dim, params_with_head)."""
    if name == 'ResNet50':
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        n_full = sum(p.numel() for p in m.parameters())
        dim = m.fc.in_features; m.fc = nn.Identity(); target = m.layer4[-1]
    elif name == 'EfficientNetB0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        n_full = sum(p.numel() for p in m.parameters())
        dim = m.classifier[-1].in_features; m.classifier = nn.Identity(); target = m.features[-1]
    elif name == 'MobileNetV2':
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None)
        n_full = sum(p.numel() for p in m.parameters())
        dim = m.classifier[-1].in_features; m.classifier = nn.Identity(); target = m.features[-1]
    else:
        raise ValueError(name)
    m.eval()
    if to_device:
        m.to(DEVICE)
    return m, target, dim, n_full


@torch.no_grad()
def extract_features(model, modality, indices=None, batch_size=None):
    idx = np.arange(N) if indices is None else np.asarray(indices)
    bs = batch_size or CONFIG['batch_size']
    out = []
    for s in range(0, len(idx), bs):
        x = arrays_to_tensor(get_images(idx[s:s + bs], modality)).to(DEVICE)
        out.append(model(x).float().cpu().numpy())
    return np.concatenate(out)


BACKBONES, FEATS, PARAMS = {}, {}, {}
for cnn in CNN_NAMES:
    model, target, dim, n_full = build_backbone(cnn, CONFIG['pretrained'])
    BACKBONES[cnn] = (model, target)
    PARAMS[cnn] = {'params_full_M': n_full / 1e6,
                   'params_backbone_M': sum(p.numel() for p in model.parameters()) / 1e6,
                   'feature_dim': dim}
    for mod in MODALITIES:
        fpath = os.path.join(DIRS['features'], f'{cnn}_{mod}.npy')
        if os.path.exists(fpath) and not CONFIG['force_recompute_features']:
            FEATS[(cnn, mod)] = np.load(fpath)
            src = 'loaded'
        else:
            t0 = time.perf_counter()
            FEATS[(cnn, mod)] = extract_features(model, mod)
            np.save(fpath, FEATS[(cnn, mod)])
            src = f'extracted in {time.perf_counter() - t0:.1f}s'
        assert FEATS[(cnn, mod)].shape == (N, dim)
        print(f'{cnn:>15} | {mod:<9} | {FEATS[(cnn, mod)].shape} | {src}')

### 3.1 Near-duplicate detection and grouping

The dataset contains images that are visually identical although their files differ (different names and
MD5 hashes). Such copies would leak information if one copy were in the training set and another in the
test set. Near-duplicates are therefore detected from the cosine similarity of the frozen MobileNetV2 RGB
features: two images are linked when their similarity is ≥ `dup_threshold` (0.99). The similarity
distribution has a clear gap below this value (see the figure). Linked images form groups (connected
components); every group is kept entirely within one subset in all splits and CV folds. The groups are
confirmed with EfficientNetB0 features and with a pixel-level difference.

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

THR = CONFIG['dup_threshold']
Fm = FEATS[(CONFIG['dup_feature'], 'RGB')]; Fm = Fm / np.linalg.norm(Fm, axis=1, keepdims=True)
Fe = FEATS[('EfficientNetB0', 'RGB')]; Fe = Fe / np.linalg.norm(Fe, axis=1, keepdims=True)
S_all = Fm @ Fm.T
np.fill_diagonal(S_all, -1)
n_groups, GROUPS = connected_components(csr_matrix(S_all >= THR), directed=False)
gsize = np.bincount(GROUPS)
nn_all = S_all.max(1)
pairs = np.argwhere(np.triu(S_all >= THR, 1))
thumbs = np.stack([np.asarray(Image.fromarray(g).resize((64, 64), Image.BILINEAR), dtype=float) for g in IMG_GRAY])
pair_mae = np.array([np.abs(thumbs[a] - thumbs[b]).mean() for a, b in pairs]) if len(pairs) else np.array([])
pair_eff = np.array([Fe[a] @ Fe[b] for a, b in pairs]) if len(pairs) else np.array([])
nn_arg = S_all.argmax(1)
ref_mae = np.array([np.abs(thumbs[i] - thumbs[nn_arg[i]]).mean() for i in np.where(nn_all < THR)[0]])

g_lab = pd.DataFrame({'g': GROUPS, 'y': LABELS}).groupby('g').y.agg(['nunique', 'size'])
cross_class = g_lab[g_lab['nunique'] > 1]
dup_groups = g_lab[g_lab['size'] > 1]
DUP_INFO = {
    'threshold': THR, 'n_images': int(N), 'n_groups': int(n_groups),
    'n_duplicate_groups': int(len(dup_groups)), 'n_images_in_duplicate_groups': int(dup_groups['size'].sum()),
    'pct_images_in_duplicate_groups': float(dup_groups['size'].sum() / N * 100),
    'group_size_distribution': {int(k): int(v) for k, v in pd.Series(gsize).value_counts().sort_index().items()},
    'n_duplicate_pairs': int(len(pairs)),
    'n_cross_class_groups': int(len(cross_class)),
    'n_images_in_cross_class_groups': int(cross_class['size'].sum()) if len(cross_class) else 0,
    'max_nn_similarity_below_threshold': float(nn_all[nn_all < THR].max()),
    'min_similarity_of_linked_pairs': float(S_all[pairs[:, 0], pairs[:, 1]].min()) if len(pairs) else None,
    'efficientnet_similarity_of_linked_pairs_min': float(pair_eff.min()) if len(pairs) else None,
    'pixel_MAE_linked_pairs_mean': float(pair_mae.mean()) if len(pairs) else None,
    'pixel_MAE_linked_pairs_max': float(pair_mae.max()) if len(pairs) else None,
    'pixel_MAE_nearest_non_duplicate_mean': float(ref_mae.mean()),
}
for k, v in DUP_INFO.items():
    print(f'{k:>45}: {v}')
dup_tab = pd.DataFrame([{'index_a': a, 'index_b': b, 'file_a': file_df.file[a], 'file_b': file_df.file[b],
                         'class_a': CLASS_NAMES[LABELS[a]], 'class_b': CLASS_NAMES[LABELS[b]],
                         'cos_mobilenetv2': S_all[a, b], 'cos_efficientnetb0': Fe[a] @ Fe[b],
                         'pixel_MAE_64px': np.abs(thumbs[a] - thumbs[b]).mean()} for a, b in pairs])
dup_tab.to_csv(os.path.join(DIRS['splits'], 'near_duplicate_pairs.csv'), index=False)
pd.DataFrame({'index': np.arange(N), 'file': file_df.file, 'label': LABELS, 'group': GROUPS,
              'group_size': gsize[GROUPS]}).to_csv(os.path.join(DIRS['splits'], 'duplicate_groups.csv'), index=False)
if len(cross_class):
    print('\nWARNING: near-duplicate groups containing both classes (possible label noise):')
    print(dup_tab[dup_tab.class_a != dup_tab.class_b].to_string(index=False))

fig = plt.figure(figsize=(16, 7))
ax = fig.add_subplot(2, 1, 1)
ax.hist(nn_all, bins=80, color='C0')
ax.axvline(THR, color='r', ls='--', label=f'threshold = {THR}')
ax.set_yscale('log'); ax.set_xlabel('Cosine similarity to the most similar other image (MobileNetV2 features)')
ax.set_ylabel('Number of images (log)'); ax.legend()
ax.set_title(f'Near-duplicate detection: {DUP_INFO["n_images_in_duplicate_groups"]} of {N} images '
             f'({DUP_INFO["pct_images_in_duplicate_groups"]:.1f} %) in {DUP_INFO["n_duplicate_groups"]} duplicate groups')
show_pairs = pairs[:4] if len(pairs) else []
for j, (a, b) in enumerate(show_pairs):
    a1 = fig.add_subplot(2, 8, 9 + 2 * j); a1.imshow(IMG_RGB[a]); a1.axis('off')
    a1.set_title(f'#{a} {CLASS_NAMES[LABELS[a]]}', fontsize=8)
    a2 = fig.add_subplot(2, 8, 10 + 2 * j); a2.imshow(IMG_RGB[b]); a2.axis('off')
    a2.set_title(f'#{b} {CLASS_NAMES[LABELS[b]]}\ncos={S_all[a, b]:.4f}', fontsize=8)
plt.tight_layout(); save_fig('fig_near_duplicate_groups.png'); plt.show()
del S_all

## 4. Group-aware data splits and cross-validation folds

Splits are made at the **group level**: all copies of a near-duplicate group are always placed in the same
subset. Groups are split with stratification on the group label, so the class ratio and the 70 / 15 / 15
proportions are preserved (subset sizes can differ by a few images between seeds). The repeated CV uses
`StratifiedGroupKFold` with a different shuffle seed for each repeat.

In [ ]:
_glab = pd.DataFrame({'g': GROUPS, 'y': LABELS}).groupby('g').y.agg(lambda v: np.bincount(v, minlength=2).argmax())


def group_split(indices, frac, seed):
    """Split `indices` into (A, B) with ~frac of the images in B; groups are never divided."""
    indices = np.asarray(indices)
    ug = np.unique(GROUPS[indices])
    g_a, g_b = train_test_split(ug, test_size=frac, stratify=_glab.loc[ug].values, random_state=seed)
    in_b = np.isin(GROUPS[indices], g_b)
    return np.sort(indices[~in_b]), np.sort(indices[in_b])


def make_split(seed):
    tv, te = group_split(np.arange(N), CONFIG['test_frac'], seed)
    tr, va = group_split(tv, CONFIG['val_frac'] / (1 - CONFIG['test_frac']), seed)
    return {'train': tr, 'val': va, 'test': te}


SPLITS = {s: make_split(s) for s in SEEDS}
split_tab = pd.DataFrame({'index': np.arange(N), 'file': file_df['file'], 'label': LABELS, 'group': GROUPS})
for s, sp in SPLITS.items():
    col = np.empty(N, dtype=object)
    for part, idx in sp.items():
        col[idx] = part
    split_tab[f'seed_{s}'] = col
    for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
        assert len(np.intersect1d(GROUPS[sp[a]], GROUPS[sp[b]])) == 0, (s, a, b)
split_tab.to_csv(os.path.join(DIRS['splits'], 'split_indices.csv'), index=False)

DEV_IDX = np.sort(np.concatenate([SPLITS[MAIN_SEED]['train'], SPLITS[MAIN_SEED]['val']]))
CV_FOLDS = []
for r in range(CONFIG['cv_repeats']):
    sgkf = StratifiedGroupKFold(n_splits=CONFIG['cv_splits'], shuffle=True, random_state=MAIN_SEED + r)
    for a, b in sgkf.split(DEV_IDX, LABELS[DEV_IDX], GROUPS[DEV_IDX]):
        assert len(np.intersect1d(GROUPS[DEV_IDX[a]], GROUPS[DEV_IDX[b]])) == 0
        CV_FOLDS.append((DEV_IDX[a], DEV_IDX[b]))
CV_NTRAIN = np.mean([len(a) for a, _ in CV_FOLDS])
CV_NTEST = np.mean([len(b) for _, b in CV_FOLDS])
fold_tab = pd.DataFrame({'index': DEV_IDX})
for fi, (_, te) in enumerate(CV_FOLDS):
    fold_tab[f'fold_{fi:02d}'] = np.isin(DEV_IDX, te).astype(int)
fold_tab.to_csv(os.path.join(DIRS['splits'], 'cv_fold_membership_dev_set.csv'), index=False)

sp0 = SPLITS[MAIN_SEED]
size_rows = []
for s, sp in SPLITS.items():
    row = {'seed': s}
    for part in ['train', 'val', 'test']:
        cnt = np.bincount(LABELS[sp[part]], minlength=2)
        row[f'{part}_n'] = len(sp[part]); row[f'{part}_{CLASS_NAMES[0]}'] = cnt[0]; row[f'{part}_{CLASS_NAMES[1]}'] = cnt[1]
    size_rows.append(row)
size_df = pd.DataFrame(size_rows)
save_table(size_df, 'table_split_sizes.csv')
print('Group-aware split sizes per seed:')
print(size_df.to_string(index=False))
print(f'Development set: {len(DEV_IDX)} images → {len(CV_FOLDS)} CV folds '
      f'(≈{CV_NTRAIN:.0f} train / {CV_NTEST:.0f} validation per fold)')
assert len(np.intersect1d(DEV_IDX, sp0['test'])) == 0

## 5. Verification: similarity between test and training images after group-aware splitting

The dataset provides no specimen / batch identifiers, so a specimen-level split is not possible. After
group-aware splitting, the cosine similarity of each test image to its most similar training image is
compared with the nearest-neighbour similarity *within* the training set (MobileNetV2 RGB features,
seed 42). No test image should have a training neighbour above the duplicate threshold.

In [ ]:
F_dup = FEATS[('MobileNetV2', 'RGB')]
F_dup = F_dup / np.linalg.norm(F_dup, axis=1, keepdims=True)
tr0, te0 = sp0['train'], sp0['test']
S_tt = F_dup[te0] @ F_dup[tr0].T
nn_test = S_tt.max(1); nn_test_arg = tr0[S_tt.argmax(1)]
S_rr = F_dup[tr0] @ F_dup[tr0].T
np.fill_diagonal(S_rr, -1)
nn_train = S_rr.max(1)

dup_rows = []
for thr in [0.90, 0.95, 0.98, 0.99]:
    dup_rows.append({'cosine_threshold': thr,
                     'test_images_with_train_neighbor_above_thr': int((nn_test > thr).sum()),
                     'pct_test': (nn_test > thr).mean() * 100,
                     'pct_train_within_train': (nn_train > thr).mean() * 100})
dup_df = pd.DataFrame(dup_rows)
save_table(dup_df, 'table_near_duplicate_check.csv')
print(dup_df.round(2).to_string(index=False))
print(f'Test→train NN similarity: median {np.median(nn_test):.4f}, max {nn_test.max():.4f}')
print(f'Train→train NN similarity: median {np.median(nn_train):.4f}, max {nn_train.max():.4f}')
ks = stats.ks_2samp(nn_test, nn_train)
print(f'KS test (test-to-train vs. train-to-train NN similarity): D={ks.statistic:.3f}, p={ks.pvalue:.3g}')

fig = plt.figure(figsize=(16, 7))
ax = fig.add_subplot(2, 1, 1)
ax.hist(nn_train, bins=60, alpha=0.6, density=True, label='train → nearest other train image')
ax.hist(nn_test, bins=60, alpha=0.6, density=True, label='test → nearest train image')
ax.set_xlabel('Cosine similarity (MobileNetV2 features)'); ax.set_ylabel('Density'); ax.legend()
ax.set_title('Nearest-neighbour similarity: test-to-train vs. within-train')
top = np.argsort(-nn_test)[:4]
for j, t in enumerate(top):
    a1 = fig.add_subplot(2, 8, 9 + 2 * j); a1.imshow(IMG_RGB[te0[t]]); a1.axis('off')
    a1.set_title(f'test #{te0[t]}', fontsize=8)
    a2 = fig.add_subplot(2, 8, 10 + 2 * j); a2.imshow(IMG_RGB[nn_test_arg[t]]); a2.axis('off')
    a2.set_title(f'train #{nn_test_arg[t]}\ncos={nn_test[t]:.4f}', fontsize=8)
plt.tight_layout(); save_fig('fig_near_duplicate_check.png'); plt.show()

## 6. Evaluation utilities

**Hybrid pipeline (single CNN):** `StandardScaler → PCA(95 % variance) → classifier`, all fitted on the
training data only.

**Classifiers:** SVM (RBF kernel, C = 10, γ = 'scale' = 1 / (n_features · Var(X))), Random Forest
(200 trees), KNN (k = 5, Euclidean distance, distance weighting).

**Continuous scores** for ROC-AUC: SVM decision function; RF / KNN predicted probability of Siirt.

In [ ]:
def make_classifier(name, seed, probability=False):
    if name == 'SVM':
        return SVC(kernel='rbf', C=CONFIG['svm_C'], gamma=CONFIG['svm_gamma'],
                   probability=probability, random_state=seed)
    if name == 'RF':
        return RandomForestClassifier(n_estimators=CONFIG['rf_n'], random_state=seed, n_jobs=-1)
    if name == 'KNN':
        return KNeighborsClassifier(n_neighbors=CONFIG['knn_k'], metric='euclidean', weights='distance')
    raise ValueError(name)


class BlockWeightedScaler(BaseEstimator, TransformerMixin):
    """Standardizes each feature block separately and multiplies it by a scalar weight.

    x_fused = [w_1 * z_1, ..., w_M * z_M],  z_m = (f_m - mu_m) / sigma_m  (mu, sigma from training data)
    No further standardization follows, so the weights are not cancelled.
    """
    def __init__(self, block_sizes=(1,), weights=None):
        self.block_sizes = block_sizes
        self.weights = weights

    def fit(self, X, y=None):
        self.bounds_ = np.cumsum([0] + list(self.block_sizes))
        w = np.ones(len(self.block_sizes)) if self.weights is None else np.asarray(self.weights, float)
        self.w_ = w
        self.scalers_ = [StandardScaler().fit(X[:, a:b]) for a, b in zip(self.bounds_[:-1], self.bounds_[1:])]
        return self

    def transform(self, X):
        return np.hstack([sc.transform(X[:, a:b]) * w for sc, w, a, b in
                          zip(self.scalers_, self.w_, self.bounds_[:-1], self.bounds_[1:])])


def make_pipeline(clf_name, seed, block_sizes=None, weights=None, probability=False):
    scaler = StandardScaler() if block_sizes is None else BlockWeightedScaler(tuple(block_sizes), weights)
    return Pipeline([('scaler', scaler),
                     ('pca', PCA(n_components=CONFIG['pca_var'], svd_solver='full', random_state=seed)),
                     ('clf', make_classifier(clf_name, seed, probability))])


def model_scores(pipe, X):
    clf = pipe.steps[-1][1]
    if isinstance(clf, SVC) and not clf.probability:
        return pipe.decision_function(X)
    return pipe.predict_proba(X)[:, 1]


def compute_metrics(y, yhat, score=None):
    y, yhat = np.asarray(y), np.asarray(yhat)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    m = {
        'acc': accuracy_score(y, yhat),
        'bal_acc': balanced_accuracy_score(y, yhat),
        'precision_macro': precision_score(y, yhat, average='macro', zero_division=0),
        'recall_macro': recall_score(y, yhat, average='macro', zero_division=0),
        'f1_macro': f1_score(y, yhat, average='macro', zero_division=0),
        'f1_weighted': f1_score(y, yhat, average='weighted', zero_division=0),
        'sensitivity': tp / (tp + fn) if tp + fn else np.nan,      # recall of Siirt
        'specificity': tn / (tn + fp) if tn + fp else np.nan,      # recall of Kirmizi
        'precision_siirt': tp / (tp + fp) if tp + fp else np.nan,
        'precision_kirmizi': tn / (tn + fn) if tn + fn else np.nan,
        'kappa': cohen_kappa_score(y, yhat),
        'mcc': matthews_corrcoef(y, yhat),
        'auc': np.nan, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'n_correct': int((y == yhat).sum()), 'n': int(len(y)),
    }
    if score is not None and len(np.unique(y)) == 2:
        m['auc'] = roc_auc_score(y, score)
    return m


def bootstrap_ci(y, yhat, score=None, metrics=('acc', 'bal_acc', 'f1_macro', 'mcc', 'kappa', 'auc'),
                 n_boot=None, seed=MAIN_SEED):
    n_boot = n_boot or CONFIG['n_bootstrap']
    rng_ = np.random.default_rng(seed)
    y, yhat = np.asarray(y), np.asarray(yhat)
    vals = {m: [] for m in metrics}
    for _ in range(n_boot):
        b = rng_.integers(0, len(y), len(y))
        if len(np.unique(y[b])) < 2:
            continue
        mm = compute_metrics(y[b], yhat[b], None if score is None else np.asarray(score)[b])
        for m in metrics:
            vals[m].append(mm[m])
    return {m: (np.nanpercentile(v, 2.5), np.nanpercentile(v, 97.5)) for m, v in vals.items()}


def corrected_ttest(diff, n_train=None, n_test=None):
    """Nadeau & Bengio (2003) corrected resampled t-test for repeated k-fold CV (paired differences)."""
    n_train = CV_NTRAIN if n_train is None else n_train
    n_test = CV_NTEST if n_test is None else n_test
    d = np.asarray(diff, float); J = len(d); mean = d.mean(); var = d.var(ddof=1)
    if var == 0:
        return {'mean_diff': mean, 'ci_low': mean, 'ci_high': mean, 't': 0.0 if mean == 0 else np.inf,
                'p': 1.0 if mean == 0 else 0.0, 'sd_diff': 0.0, 'J': J}
    se = np.sqrt((1.0 / J + n_test / n_train) * var)
    t = mean / se
    half = stats.t.ppf(0.975, J - 1) * se
    return {'mean_diff': mean, 'ci_low': mean - half, 'ci_high': mean + half, 't': t,
            'p': 2 * stats.t.sf(abs(t), J - 1), 'sd_diff': np.sqrt(var), 'J': J}


def corrected_ci(values, n_train=None, n_test=None):
    """Mean and corrected 95 % CI of a CV metric (same variance correction as the t-test)."""
    n_train = CV_NTRAIN if n_train is None else n_train
    n_test = CV_NTEST if n_test is None else n_test
    v = np.asarray(values, float); J = len(v)
    se = np.sqrt((1.0 / J + n_test / n_train) * v.var(ddof=1))
    half = stats.t.ppf(0.975, J - 1) * se
    return v.mean(), v.mean() - half, v.mean() + half


def mcnemar_test(y, pred_a, pred_b):
    """Exact McNemar test. b = A correct & B wrong, c = A wrong & B correct."""
    ca, cb = (pred_a == y), (pred_b == y)
    a, b, c, d = int((ca & cb).sum()), int((ca & ~cb).sum()), int((~ca & cb).sum()), int((~ca & ~cb).sum())
    res = mcnemar(np.array([[a, b], [c, d]]), exact=True)
    return {'both_correct': a, 'b_A_correct_B_wrong': b, 'c_A_wrong_B_correct': c, 'both_wrong': d,
            'statistic': float(res.statistic), 'p_exact': float(res.pvalue)}


def holm(pvals):
    p = np.asarray(pvals, float)
    return multipletests(p, method='holm')[1] if len(p) else p


PCT_METRICS = ['acc', 'bal_acc', 'f1_macro', 'f1_weighted', 'sensitivity', 'specificity',
               'precision_macro', 'recall_macro']
RAW_METRICS = ['kappa', 'mcc', 'auc']
TABLE_METRICS = ['acc', 'bal_acc', 'f1_macro', 'f1_weighted', 'sensitivity', 'specificity', 'kappa', 'mcc', 'auc']


def pm_table(df, group_cols, metrics=TABLE_METRICS, sort_by='acc'):
    """mean ± SD (ddof = 1) per group; percentages for rate metrics, 4 decimals for kappa/MCC/AUC."""
    g = df.groupby(group_cols)[metrics]
    mean, sd, n = g.mean(), g.std(ddof=1), g.size()
    out = pd.DataFrame(index=mean.index)
    for m in metrics:
        if m in PCT_METRICS:
            out[m] = [f'{a * 100:.2f} ± {b * 100:.2f}' for a, b in zip(mean[m], sd[m].fillna(0))]
        else:
            out[m] = [f'{a:.4f} ± {b:.4f}' for a, b in zip(mean[m], sd[m].fillna(0))]
    out['n_runs'] = n
    out['_sort'] = mean[sort_by]
    return out.sort_values('_sort', ascending=False).drop(columns='_sort')


SINGLE_MODELS = [(f'{mod}+{cnn}+{clf}', mod, cnn, clf)
                 for mod in MODALITIES for cnn in CNN_NAMES for clf in CLF_NAMES]
PRED = {}   # (experiment, model, seed) -> dict(idx, y_true, y_pred, score)


def store_pred(exp, model, seed, idx, yhat, score):
    PRED[(exp, model, seed)] = {'idx': np.asarray(idx), 'y_true': LABELS[idx],
                                'y_pred': np.asarray(yhat), 'score': np.asarray(score)}


print(f'{len(SINGLE_MODELS)} single-CNN hybrid combinations defined.')

## 7. Computational cost

Parameters are reported for the complete ImageNet model and for the backbone without its classification
head (the part used here). GPU inference time is measured with CUDA synchronization after 20 warm-up
passes, FP32, averaged over `timing_runs` forward passes, for batch sizes 1 and 32. The time of the
scaler + PCA + SVM step (seed-42 pipeline, one image at a time, CPU) is reported separately.

In [ ]:
def sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


@torch.no_grad()
def time_backbone(model, bs, n_warmup=20, n_runs=None):
    n_runs = n_runs or CONFIG['timing_runs']
    x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    for _ in range(n_warmup):
        model(x)
    sync(); t0 = time.perf_counter()
    for _ in range(n_runs):
        model(x)
    sync()
    return (time.perf_counter() - t0) / n_runs / bs * 1000


cost_rows = []
for cnn in CNN_NAMES:
    model, _ = BACKBONES[cnn]
    X = FEATS[(cnn, 'RGB')]
    pipe = make_pipeline('SVM', MAIN_SEED).fit(X[sp0['train']], LABELS[sp0['train']])
    Xte = X[sp0['test']]
    t0 = time.perf_counter()
    for i in range(len(Xte)):
        pipe.predict(Xte[i:i + 1])
    head_ms = (time.perf_counter() - t0) / len(Xte) * 1000
    bb1, bb32 = time_backbone(model, 1), time_backbone(model, 32)
    cost_rows.append({'Model': cnn, 'Params full model (M)': round(PARAMS[cnn]['params_full_M'], 2),
                      'Params backbone (M)': round(PARAMS[cnn]['params_backbone_M'], 2),
                      'Feature dim': PARAMS[cnn]['feature_dim'],
                      'PCA dim (seed 42)': int(pipe.named_steps['pca'].n_components_),
                      'Backbone ms/img (bs=1)': round(bb1, 2), 'Backbone ms/img (bs=32)': round(bb32, 2),
                      'Scaler+PCA+SVM ms/img': round(head_ms, 3),
                      'Total ms/img (bs=1)': round(bb1 + head_ms, 2)})
cost_df = pd.DataFrame(cost_rows)
save_table(cost_df, 'table_computational_cost.csv')
print(f"Hardware: {ENV['gpu'] or ENV['processor']} | device={DEVICE} | FP32 | warm-up=20 | runs={CONFIG['timing_runs']}")
print(cost_df.to_string(index=False))

## 8. Experiment 1 — 18 hybrid combinations, hold-out evaluation with 5 seeds

For every seed the pipeline is fitted on the training set; validation and test sets are only predicted.
Train, validation and test metrics are stored for every run.

In [ ]:
records, pca_rows = [], []
for seed in tqdm(SEEDS, desc='Seeds'):
    sp = SPLITS[seed]
    for key, mod, cnn, clf in SINGLE_MODELS:
        X = FEATS[(cnn, mod)]
        pipe = make_pipeline(clf, seed)
        t0 = time.perf_counter(); pipe.fit(X[sp['train']], LABELS[sp['train']]); fit_s = time.perf_counter() - t0
        pca_rows.append({'seed': seed, 'model': key, 'cnn': cnn, 'modality': mod,
                         'raw_dim': X.shape[1], 'pca_dim': int(pipe.named_steps['pca'].n_components_)})
        for part in ['train', 'val', 'test']:
            idx = sp[part]
            yhat = pipe.predict(X[idx]); sc = model_scores(pipe, X[idx])
            records.append({'seed': seed, 'model': key, 'modality': mod, 'cnn': cnn, 'classifier': clf,
                            'part': part, 'fit_s': fit_s, **compute_metrics(LABELS[idx], yhat, sc)})
            if part == 'test':
                store_pred('single', key, seed, idx, yhat, sc)
HOLD = pd.DataFrame(records)
HOLD.to_csv(os.path.join(DIRS['tables'], 'raw_holdout_single_models.csv'), index=False)

pca_df = pd.DataFrame(pca_rows)
pca_sum = pca_df.groupby(['cnn', 'modality']).agg(raw_dim=('raw_dim', 'first'), pca_dim_mean=('pca_dim', 'mean'),
                                                  pca_dim_sd=('pca_dim', 'std'), pca_dim_min=('pca_dim', 'min'),
                                                  pca_dim_max=('pca_dim', 'max')).round(1)
save_table(pca_sum, 'table_pca_dimensions.csv', index=True)
print('Feature dimensionality before / after PCA (95 % variance):')
print(pca_sum.to_string())

hold_test = pm_table(HOLD[HOLD.part == 'test'], ['model'])
hold_val = pm_table(HOLD[HOLD.part == 'val'], ['model'], metrics=['acc', 'f1_macro', 'mcc'])
save_table(hold_test, 'table_holdout_test_all18_mean_sd.csv', index=True)
save_table(hold_val, 'table_holdout_val_all18_mean_sd.csv', index=True)
print(f'\nHold-out TEST set, mean ± SD over {len(SEEDS)} seeds (all 18 combinations):')
print(hold_test.to_string())

seed42 = HOLD[(HOLD.part == 'test') & (HOLD.seed == MAIN_SEED)].set_index('model')
seed42_tab = seed42[['acc', 'f1_macro', 'kappa', 'mcc', 'auc', 'n_correct', 'n']].sort_values('acc', ascending=False)
save_table(seed42_tab, 'table_holdout_test_seed42_all18.csv', index=True)

## 9. Experiment 2 — repeated 5 × 5 cross-validation on the development set and statistical comparison

All 18 combinations are evaluated on exactly the same 25 folds, so results are paired. Because CV folds
share training data, the Friedman / Nemenyi procedure (designed for independent data sets) is not
applied. Instead, pairwise differences are tested with the **Nadeau–Bengio corrected resampled t-test**
(variance multiplied by 1/J + n_test/n_train, J = 25, df = 24), with **Holm** correction within each
family of comparisons. Effect sizes are reported as mean differences in percentage points with
corrected 95 % CIs.

In [ ]:
cv_records = []
for fi, (tr, te) in enumerate(tqdm(CV_FOLDS, desc='CV folds')):
    for key, mod, cnn, clf in SINGLE_MODELS:
        X = FEATS[(cnn, mod)]
        pipe = make_pipeline(clf, MAIN_SEED).fit(X[tr], LABELS[tr])
        yhat = pipe.predict(X[te]); sc = model_scores(pipe, X[te])
        m = compute_metrics(LABELS[te], yhat, sc)
        m['train_acc'] = accuracy_score(LABELS[tr], pipe.predict(X[tr]))
        cv_records.append({'fold': fi, 'repeat': fi // CONFIG['cv_splits'], 'model': key, 'modality': mod,
                           'cnn': cnn, 'classifier': clf, **m})
CV = pd.DataFrame(cv_records)
CV.to_csv(os.path.join(DIRS['tables'], 'raw_cv_single_models.csv'), index=False)

cv_rows = []
for key, mod, cnn, clf in SINGLE_MODELS:
    s = CV[CV.model == key]
    mean, lo, hi = corrected_ci(s.acc)
    cv_rows.append({'model': key, 'modality': mod, 'cnn': cnn, 'classifier': clf,
                    'cv_acc_mean': mean, 'cv_acc_sd': s.acc.std(ddof=1), 'cv_acc_ci_low': lo, 'cv_acc_ci_high': hi,
                    'cv_f1_macro_mean': s.f1_macro.mean(), 'cv_f1_macro_sd': s.f1_macro.std(ddof=1),
                    'cv_mcc_mean': s.mcc.mean(), 'cv_mcc_sd': s.mcc.std(ddof=1),
                    'cv_kappa_mean': s.kappa.mean(), 'cv_auc_mean': s.auc.mean(), 'cv_auc_sd': s.auc.std(ddof=1),
                    'cv_train_acc_mean': s.train_acc.mean(),
                    'train_minus_val_gap_pp': (s.train_acc.mean() - s.acc.mean()) * 100})
CV_SUM = pd.DataFrame(cv_rows).sort_values('cv_acc_mean', ascending=False).reset_index(drop=True)
CV_SUM['rank'] = np.arange(1, len(CV_SUM) + 1)
save_table(CV_SUM.round(5), 'table_cv_all18_summary.csv')
print('Repeated 5x5 CV on the development set (mean, SD, corrected 95 % CI):')
show = CV_SUM[['rank', 'model', 'cv_acc_mean', 'cv_acc_sd', 'cv_acc_ci_low', 'cv_acc_ci_high',
               'cv_f1_macro_mean', 'cv_mcc_mean', 'cv_auc_mean', 'cv_train_acc_mean']].copy()
for c in ['cv_acc_mean', 'cv_acc_sd', 'cv_acc_ci_low', 'cv_acc_ci_high', 'cv_f1_macro_mean', 'cv_train_acc_mean']:
    show[c] = (show[c] * 100).round(2)
print(show.to_string(index=False))

### 9.1 Model selection (development data only)
The model with the highest mean CV accuracy is selected. Its validation-set ranking in the hold-out
experiment is shown for comparison. Test results are not used for selection.

In [ ]:
BEST_KEY = CV_SUM.iloc[0]['model']
BEST_MOD, BEST_CNN, BEST_CLF = BEST_KEY.split('+')
val_rank = (HOLD[HOLD.part == 'val'].groupby('model').acc.mean().sort_values(ascending=False))
print(f'Selected model (highest CV accuracy): {BEST_KEY}')
print(f'  CV accuracy: {CV_SUM.iloc[0].cv_acc_mean * 100:.2f} ± {CV_SUM.iloc[0].cv_acc_sd * 100:.2f} % '
      f'(95 % CI {CV_SUM.iloc[0].cv_acc_ci_low * 100:.2f}–{CV_SUM.iloc[0].cv_acc_ci_high * 100:.2f})')
print(f'  Rank by mean hold-out validation accuracy: {list(val_rank.index).index(BEST_KEY) + 1} '
      f'(top-3 by validation: {list(val_rank.index[:3])})')

### 9.2 Pairwise statistical comparisons (corrected resampled t-test, Holm)

In [ ]:
ACC_MAT = CV.pivot(index='fold', columns='model', values='acc')
order = list(CV_SUM.model)


def compare_pairs(pairs, family, mat=ACC_MAT):
    rows = []
    for a, b in pairs:
        r = corrected_ttest(mat[a].values - mat[b].values)
        rows.append({'family': family, 'model_A': a, 'model_B': b,
                     'mean_A_%': mat[a].mean() * 100, 'mean_B_%': mat[b].mean() * 100,
                     'diff_pp': r['mean_diff'] * 100, 'ci_low_pp': r['ci_low'] * 100, 'ci_high_pp': r['ci_high'] * 100,
                     'sd_diff_pp': r['sd_diff'] * 100,
                     'standardized_diff': r['mean_diff'] / r['sd_diff'] if r['sd_diff'] > 0 else np.nan,
                     't': r['t'], 'df': r['J'] - 1, 'p': r['p']})
    df = pd.DataFrame(rows)
    df['p_holm'] = holm(df.p.values)
    df['significant_0.05'] = df.p_holm < 0.05
    return df


# Family A: all 153 pairs (complete pairwise comparison table)
fam_all = compare_pairs(list(combinations(order, 2)), 'A: all pairs')
# Family B: classifiers within each CNN x modality
pairs_B = [(f'{m}+{c}+{a}', f'{m}+{c}+{b}') for m in MODALITIES for c in CNN_NAMES
           for a, b in [('SVM', 'RF'), ('SVM', 'KNN'), ('KNN', 'RF')]]
fam_clf = compare_pairs(pairs_B, 'B: classifier within CNN x modality')
# Family C: RGB vs grayscale
fam_mod = compare_pairs([(f'RGB+{c}+{k}', f'Grayscale+{c}+{k}') for c in CNN_NAMES for k in CLF_NAMES],
                        'C: RGB vs grayscale')
# Family D: CNN architectures with SVM
fam_cnn = compare_pairs([(f'{m}+{a}+SVM', f'{m}+{b}+SVM') for m in MODALITIES
                         for a, b in combinations(CNN_NAMES, 2)], 'D: CNN architecture (SVM)')
# Family E: selected model vs. every other model
fam_best = compare_pairs([(BEST_KEY, o) for o in order if o != BEST_KEY], 'E: selected model vs others')

STATS = pd.concat([fam_all, fam_clf, fam_mod, fam_cnn, fam_best], ignore_index=True)
save_table(STATS.round(6), 'table_stats_corrected_ttest_all_families.csv')
cols = ['model_A', 'model_B', 'diff_pp', 'ci_low_pp', 'ci_high_pp', 't', 'p', 'p_holm', 'significant_0.05']
for name, fam in [('Classifier comparisons (family B)', fam_clf), ('RGB vs grayscale (family C)', fam_mod),
                  ('CNN comparisons with SVM (family D)', fam_cnn), ('Selected model vs others (family E)', fam_best)]:
    print(f'\n{name}:')
    print(fam[cols].round(4).to_string(index=False))

n_sig = fam_all['significant_0.05'].sum()
print(f'\nFamily A: {n_sig} of {len(fam_all)} pairs significant after Holm correction.')

P = pd.DataFrame(np.nan, index=order, columns=order)
for _, r in fam_all.iterrows():
    P.loc[r.model_A, r.model_B] = P.loc[r.model_B, r.model_A] = r.p_holm
D = pd.DataFrame(np.nan, index=order, columns=order)
for _, r in fam_all.iterrows():
    D.loc[r.model_A, r.model_B] = r.diff_pp; D.loc[r.model_B, r.model_A] = -r.diff_pp
short = [o.replace('Grayscale', 'Gray').replace('EfficientNetB0', 'EffNetB0').replace('MobileNetV2', 'MobNetV2')
         for o in order]
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
sns.heatmap(P.values, ax=axes[0], cmap='RdYlGn_r', vmin=0, vmax=0.1, xticklabels=short, yticklabels=short,
            annot=True, fmt='.2f', annot_kws={'size': 6}, cbar_kws={'label': 'Holm-adjusted p'})
axes[0].set_title('Pairwise corrected resampled t-test (Holm-adjusted p, all 153 pairs)')
sns.heatmap(D.values, ax=axes[1], cmap='RdBu', center=0, xticklabels=short, yticklabels=short,
            annot=True, fmt='.1f', annot_kws={'size': 6}, cbar_kws={'label': 'row − column accuracy (pp)'})
axes[1].set_title('Mean CV accuracy difference (percentage points)')
for ax in axes:
    ax.tick_params(axis='x', rotation=90, labelsize=7); ax.tick_params(axis='y', labelsize=7)
plt.tight_layout(); save_fig('fig_pairwise_stats_heatmap.png'); plt.show()

fig, ax = plt.subplots(figsize=(16, 5))
pal = {'SVM': '#1f77b4', 'RF': '#2ca02c', 'KNN': '#ff7f0e'}
cvp = CV.copy(); cvp['acc_%'] = cvp.acc * 100
sns.boxplot(data=cvp, x='model', y='acc_%', order=order, hue='classifier', dodge=False, palette=pal, ax=ax)
ax.set_xticklabels(short, rotation=60, ha='right', fontsize=8)
ax.set_ylabel('Accuracy (%)'); ax.set_xlabel('')
ax.set_title('Repeated 5 × 5 CV accuracy of all 18 combinations (development set, sorted by mean)')
plt.tight_layout(); save_fig('fig_cv_all18_boxplot.png'); plt.show()

## 10. Detailed results of the selected model and of all combinations (hold-out test sets)

In [ ]:
bp = PRED[('single', BEST_KEY, MAIN_SEED)]
best_m = compute_metrics(bp['y_true'], bp['y_pred'], bp['score'])
best_ci = bootstrap_ci(bp['y_true'], bp['y_pred'], bp['score'])
print(f'Selected model {BEST_KEY}, seed-{MAIN_SEED} test set (n = {best_m["n"]}): '
      f'{best_m["n_correct"]}/{best_m["n"]} correct')
rows = []
for m in ['acc', 'bal_acc', 'f1_macro', 'kappa', 'mcc', 'auc']:
    rows.append({'metric': m, 'value': best_m[m], 'bootstrap_95CI_low': best_ci[m][0], 'bootstrap_95CI_high': best_ci[m][1]})
for m in ['sensitivity', 'specificity', 'precision_siirt', 'precision_kirmizi', 'f1_weighted']:
    rows.append({'metric': m, 'value': best_m[m]})
best_tab = pd.DataFrame(rows)
save_table(best_tab, 'table_selected_model_seed42_test_with_CI.csv')
print(best_tab.round(4).to_string(index=False))
print('\nPer-class report (seed 42 test set):')
print(classification_report(bp['y_true'], bp['y_pred'], target_names=CLASS_NAMES, digits=4))

bh = HOLD[(HOLD.part == 'test') & (HOLD.model == BEST_KEY)]
print(f'Selected model over 5 seeds (test): acc {bh.acc.mean() * 100:.2f} ± {bh.acc.std(ddof=1) * 100:.2f} %, '
      f'macro-F1 {bh.f1_macro.mean() * 100:.2f} ± {bh.f1_macro.std(ddof=1) * 100:.2f} %, '
      f'MCC {bh.mcc.mean():.4f} ± {bh.mcc.std(ddof=1):.4f}, AUC {bh.auc.mean():.4f} ± {bh.auc.std(ddof=1):.4f}')

In [ ]:
# Confusion matrices of all 18 combinations (seed-42 test set; values from the stored predictions)
fig, axes = plt.subplots(3, 6, figsize=(24, 12))
for ax, (key, mod, cnn, clf) in zip(axes.T.ravel(), SINGLE_MODELS):
    p = PRED[('single', key, MAIN_SEED)]
    cm = confusion_matrix(p['y_true'], p['y_pred'], labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f'{mod} | {cnn} + {clf}\nAcc = {np.trace(cm)}/{cm.sum()} = {np.trace(cm) / cm.sum() * 100:.2f} %',
                 fontsize=9)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.suptitle(f'Confusion matrices — test set (seed {MAIN_SEED}, n = {len(sp0["test"])})', fontweight='bold')
plt.tight_layout(); save_fig('fig_confusion_matrices_all18.png'); plt.show()

In [ ]:
# ROC curves (all 18) and ROC / PR of the best RGB and best grayscale model (selected by CV)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
cmap = plt.get_cmap('tab10')
for ax, mod in zip(axes, MODALITIES):
    for i, (key, m_, cnn, clf) in enumerate([s for s in SINGLE_MODELS if s[1] == mod]):
        p = PRED[('single', key, MAIN_SEED)]
        fpr, tpr, _ = roc_curve(p['y_true'], p['score'])
        ax.plot(fpr, tpr, color=cmap(i), lw=1.5, ls='-' if clf == 'SVM' else ('--' if clf == 'KNN' else ':'),
                label=f'{cnn}+{clf} (AUC = {roc_auc_score(p["y_true"], p["score"]):.4f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
    ax.set_title(f'ROC — {mod} (test set, seed {MAIN_SEED}; positive class = Siirt)')
    ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.legend(fontsize=8, loc='lower right')
plt.tight_layout(); save_fig('fig_roc_all18.png'); plt.show()

best_by_mod = {mod: CV_SUM[CV_SUM.modality == mod].iloc[0].model for mod in MODALITIES}
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for j, mod in enumerate(MODALITIES):
    key = best_by_mod[mod]; p = PRED[('single', key, MAIN_SEED)]
    fpr, tpr, _ = roc_curve(p['y_true'], p['score']); a = roc_auc_score(p['y_true'], p['score'])
    axes[0, j].plot(fpr, tpr, lw=2, label=f'AUC = {a:.4f}'); axes[0, j].plot([0, 1], [0, 1], 'k--', lw=0.8)
    axes[0, j].set_title(f'ROC — {key}'); axes[0, j].legend(loc='lower right')
    axes[0, j].set_xlabel('False positive rate'); axes[0, j].set_ylabel('True positive rate')
    pr, rc, _ = precision_recall_curve(p['y_true'], p['score']); ap = average_precision_score(p['y_true'], p['score'])
    axes[1, j].plot(rc, pr, lw=2, color='C3', label=f'AP = {ap:.4f}')
    axes[1, j].axhline(p['y_true'].mean(), color='k', ls='--', lw=0.8, label='Prevalence')
    axes[1, j].set_title(f'Precision–recall — {key}'); axes[1, j].legend(loc='lower left')
    axes[1, j].set_xlabel('Recall (Siirt)'); axes[1, j].set_ylabel('Precision (Siirt)')
plt.suptitle(f'Best RGB and best grayscale model (selected by CV), test set seed {MAIN_SEED}', fontweight='bold')
plt.tight_layout(); save_fig('fig_roc_pr_best_models.png'); plt.show()

## 11. Split-consistency and overfitting analysis
Training accuracy, CV validation accuracy, hold-out validation accuracy and hold-out test accuracy are
compared quantitatively. A large train–validation gap indicates overfitting; a small validation–test
difference indicates that the test split is not unusually easy or hard (split consistency).

In [ ]:
h = HOLD.groupby(['model', 'part']).acc.mean().unstack()
ov = pd.DataFrame({'train_acc_%': h['train'] * 100, 'holdout_val_acc_%': h['val'] * 100,
                   'holdout_test_acc_%': h['test'] * 100})
ov = ov.join(CV_SUM.set_index('model')[['cv_acc_mean', 'cv_train_acc_mean']] * 100)
ov = ov.rename(columns={'cv_acc_mean': 'cv_val_acc_%', 'cv_train_acc_mean': 'cv_train_acc_%'})
ov['train_minus_cv_val_pp'] = ov['cv_train_acc_%'] - ov['cv_val_acc_%']
ov['val_minus_test_pp'] = ov['holdout_val_acc_%'] - ov['holdout_test_acc_%']
ov = ov.loc[order]
save_table(ov.round(3), 'table_overfitting_split_consistency.csv', index=True)
print(ov.round(2).to_string())
print(f"\nMean |validation − test| difference across 18 models: {ov['val_minus_test_pp'].abs().mean():.2f} pp "
      f"(max {ov['val_minus_test_pp'].abs().max():.2f} pp)")

fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(order)); w = 0.27
ax.bar(x - w, ov['cv_train_acc_%'], w, label='Train (CV)')
ax.bar(x, ov['holdout_val_acc_%'], w, label='Validation (hold-out, mean of 5 seeds)')
ax.bar(x + w, ov['holdout_test_acc_%'], w, label='Test (hold-out, mean of 5 seeds)')
ax.set_xticks(x); ax.set_xticklabels(short, rotation=60, ha='right', fontsize=8)
ax.set_ylim(max(0, ov.values[:, :5].min() - 5), 100.5); ax.set_ylabel('Accuracy (%)'); ax.legend()
ax.set_title('Training, validation and test accuracy (split-consistency check)')
plt.tight_layout(); save_fig('fig_split_consistency.png'); plt.show()

## 12. Experiment 3 — RGB + grayscale feature fusion (9 combinations)
For each CNN, RGB and grayscale feature blocks are standardized separately and concatenated with equal
weights, then PCA + classifier. Evaluated with the same 5 seeds and 25 CV folds; each fusion model is
compared with the corresponding RGB-only model (corrected t-test, Holm).

In [ ]:
RG_MODELS = [(f'RGB+Gray fusion+{cnn}+{clf}', cnn, clf) for cnn in CNN_NAMES for clf in CLF_NAMES]
X_RG = {cnn: np.hstack([FEATS[(cnn, 'RGB')], FEATS[(cnn, 'Grayscale')]]) for cnn in CNN_NAMES}
rg_hold, rg_cv = [], []
for seed in tqdm(SEEDS, desc='RGB+Gray fusion, seeds'):
    sp = SPLITS[seed]
    for key, cnn, clf in RG_MODELS:
        d = FEATS[(cnn, 'RGB')].shape[1]
        pipe = make_pipeline(clf, seed, block_sizes=[d, d]).fit(X_RG[cnn][sp['train']], LABELS[sp['train']])
        for part in ['val', 'test']:
            idx = sp[part]; yhat = pipe.predict(X_RG[cnn][idx]); sc = model_scores(pipe, X_RG[cnn][idx])
            rg_hold.append({'seed': seed, 'model': key, 'cnn': cnn, 'classifier': clf, 'part': part,
                            **compute_metrics(LABELS[idx], yhat, sc)})
            if part == 'test':
                store_pred('rgb_gray_fusion', key, seed, idx, yhat, sc)
for fi, (tr, te) in enumerate(tqdm(CV_FOLDS, desc='RGB+Gray fusion, CV')):
    for key, cnn, clf in RG_MODELS:
        d = FEATS[(cnn, 'RGB')].shape[1]
        pipe = make_pipeline(clf, MAIN_SEED, block_sizes=[d, d]).fit(X_RG[cnn][tr], LABELS[tr])
        yhat = pipe.predict(X_RG[cnn][te]); sc = model_scores(pipe, X_RG[cnn][te])
        rg_cv.append({'fold': fi, 'model': key, 'cnn': cnn, 'classifier': clf, **compute_metrics(LABELS[te], yhat, sc)})
RG_HOLD, RG_CV = pd.DataFrame(rg_hold), pd.DataFrame(rg_cv)
RG_HOLD.to_csv(os.path.join(DIRS['tables'], 'raw_holdout_rgb_gray_fusion.csv'), index=False)
RG_CV.to_csv(os.path.join(DIRS['tables'], 'raw_cv_rgb_gray_fusion.csv'), index=False)

rg_tab = pm_table(RG_HOLD[RG_HOLD.part == 'test'], ['model'])
rg_cvm = RG_CV.groupby('model').acc.agg(['mean', 'std'])
rg_tab['cv_acc'] = [f'{rg_cvm.loc[k, "mean"] * 100:.2f} ± {rg_cvm.loc[k, "std"] * 100:.2f}' for k in rg_tab.index]
save_table(rg_tab, 'table_rgb_gray_fusion_mean_sd.csv', index=True)
print('RGB + grayscale fusion — hold-out test (mean ± SD over 5 seeds) and CV accuracy:')
print(rg_tab[['acc', 'f1_macro', 'mcc', 'auc', 'cv_acc']].to_string())

RG_MAT = RG_CV.pivot(index='fold', columns='model', values='acc')
rows = []
for key, cnn, clf in RG_MODELS:
    r = corrected_ttest(RG_MAT[key].values - ACC_MAT[f'RGB+{cnn}+{clf}'].values)
    rows.append({'fusion_model': key, 'reference': f'RGB+{cnn}+{clf}', 'diff_pp': r['mean_diff'] * 100,
                 'ci_low_pp': r['ci_low'] * 100, 'ci_high_pp': r['ci_high'] * 100, 't': r['t'], 'p': r['p']})
rg_stats = pd.DataFrame(rows); rg_stats['p_holm'] = holm(rg_stats.p)
save_table(rg_stats.round(5), 'table_stats_rgb_gray_fusion_vs_rgb.csv')
print('\nFusion − RGB-only (CV accuracy, corrected t-test, Holm):')
print(rg_stats.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
plot_rows = []
for cnn in CNN_NAMES:
    for clf in CLF_NAMES:
        for lab, k, src in [('RGB', f'RGB+{cnn}+{clf}', CV), ('Grayscale', f'Grayscale+{cnn}+{clf}', CV),
                            ('RGB+Gray fusion', f'RGB+Gray fusion+{cnn}+{clf}', RG_CV)]:
            v = src[src.model == k].acc * 100
            plot_rows.append({'combo': f'{cnn}+{clf}', 'input': lab, 'mean': v.mean(), 'sd': v.std(ddof=1)})
pr_ = pd.DataFrame(plot_rows)
combos = pr_.combo.unique(); xx = np.arange(len(combos))
for i, lab in enumerate(['RGB', 'Grayscale', 'RGB+Gray fusion']):
    s = pr_[pr_.input == lab].set_index('combo').loc[combos]
    ax.bar(xx + (i - 1) * 0.27, s['mean'], 0.27, yerr=s['sd'], capsize=3, label=lab)
ax.set_xticks(xx); ax.set_xticklabels(combos, rotation=30, ha='right', fontsize=8)
ax.set_ylim(pr_['mean'].min() - 5, 100.5); ax.set_ylabel('CV accuracy (%), mean ± SD'); ax.legend()
ax.set_title('RGB vs. grayscale vs. RGB+grayscale feature fusion (repeated 5 × 5 CV)')
plt.tight_layout(); save_fig('fig_rgb_gray_fusion.png'); plt.show()

## 13. Experiment 4 — multi-CNN feature fusion (RGB, SVM)

**Validation-based weighted early fusion.** For CNN m, let f_m be its feature vector and
z_m = (f_m − μ_m) / σ_m its standardized version (μ, σ estimated on the training data).
Let a_m be the validation accuracy of the single-CNN pipeline (CNN m + StandardScaler + PCA + SVM,
fitted on training data, evaluated on validation data). The weights are

w_m = exp(a_m / τ) / Σ_j exp(a_j / τ)

and the fused vector is x = [w_1 z_1, w_2 z_2, w_3 z_3] (concatenation), followed by PCA (95 %) and
SVM (RBF, C = 10, γ = 'scale'). Weights are applied after per-block standardization and before PCA, and no
further standardization follows, so they are not cancelled. Because γ = 'scale' and the 95 % variance
criterion are invariant to a common scale factor, only the *relative* weights matter. The v4 analysis
used τ = 0.1 (`exp(10 · a_m)`).

**Strategies compared** (all decisions on validation data only):
* Single CNN (val-selected): the CNN with the highest a_m.
* Early fusion with equal weights (w = 1); with dimension-normalized weights (w_m = 1/√d_m, equal total
  variance per block); with softmax weights for τ ∈ {1, 0.1, 0.05, 0.02, 0.01}.
* Early fusion of the validation-selected CNN subset (best of the 4 subsets with ≥ 2 CNNs).
* Late fusion (soft voting of per-CNN SVM probabilities) with equal and softmax (τ = 0.1) weights.

**Protocols.** (i) 5 hold-out seeds: weights from the validation set, models fitted on the training set,
evaluation on the test set. (ii) 25 CV folds: in each fold an inner stratified 15 % validation split of
the fold's training data is used to compute a_m and the subset choice; final models are refitted on the
full fold training data and evaluated on the fold's validation data.

In [ ]:
DIMS = {c: FEATS[(c, 'RGB')].shape[1] for c in CNN_NAMES}
SUBSETS = [s for r in (2, 3) for s in combinations(CNN_NAMES, r)]


def softmax_w(a, tau):
    a = np.asarray(a, float) / tau
    e = np.exp(a - a.max())
    return e / e.sum()


def fused_X(cnns):
    return np.hstack([FEATS[(c, 'RGB')] for c in cnns])


def multicnn_fusion(fit_idx, sel_tr, sel_va, eval_idx, seed):
    y = LABELS
    acc_val = {}
    for c in CNN_NAMES:
        X = FEATS[(c, 'RGB')]
        p = make_pipeline('SVM', seed).fit(X[sel_tr], y[sel_tr])
        acc_val[c] = accuracy_score(y[sel_va], p.predict(X[sel_va]))
    a = np.array([acc_val[c] for c in CNN_NAMES])
    sub_val = {}
    for s in SUBSETS:
        X = fused_X(s)
        p = make_pipeline('SVM', seed, block_sizes=[DIMS[c] for c in s]).fit(X[sel_tr], y[sel_tr])
        sub_val[s] = accuracy_score(y[sel_va], p.predict(X[sel_va]))
    best_single = CNN_NAMES[int(np.argmax(a))]
    best_subset = max(SUBSETS, key=lambda s: (sub_val[s], len(s)))

    strategies = {'Single CNN (val-selected)': ((best_single,), [1.0], 'early'),
                  'Early fusion, equal weights': (tuple(CNN_NAMES), [1.0, 1.0, 1.0], 'early'),
                  'Early fusion, dimension-normalized': (tuple(CNN_NAMES), [1 / np.sqrt(DIMS[c]) for c in CNN_NAMES], 'early')}
    for tau in CONFIG['fusion_taus']:
        strategies[f'Early fusion, val-weighted (tau={tau:g})'] = (tuple(CNN_NAMES), list(softmax_w(a, tau)), 'early')
    strategies['Early fusion, val-selected subset'] = (best_subset, [1.0] * len(best_subset), 'early')
    strategies['Late fusion, equal weights'] = (tuple(CNN_NAMES), [1 / 3] * 3, 'late')
    strategies['Late fusion, val-weighted (tau=0.1)'] = (tuple(CNN_NAMES), list(softmax_w(a, 0.1)), 'late')

    out, late_probs = {}, None
    for name, (cnns, w, kind) in strategies.items():
        if kind == 'early':
            X = fused_X(cnns)
            p = make_pipeline('SVM', seed, block_sizes=[DIMS[c] for c in cnns], weights=w).fit(X[fit_idx], y[fit_idx])
            yhat, sc = p.predict(X[eval_idx]), model_scores(p, X[eval_idx])
        else:
            if late_probs is None:
                late_probs = {}
                for c in CNN_NAMES:
                    X = FEATS[(c, 'RGB')]
                    p = make_pipeline('SVM', seed, probability=True).fit(X[fit_idx], y[fit_idx])
                    late_probs[c] = p.predict_proba(X[eval_idx])[:, 1]
            ww = np.asarray(w) / np.sum(w)
            sc = sum(wi * late_probs[c] for wi, c in zip(ww, CNN_NAMES))
            yhat = (sc >= 0.5).astype(int)
        out[name] = (yhat, sc)
    info = {'acc_val': acc_val, 'best_single': best_single, 'best_subset': '+'.join(best_subset),
            'weights': {n: dict(zip(s[0], np.round(s[1], 6))) for n, s in strategies.items()}}
    return out, info


mf_hold, mf_cv, mf_info = [], [], []
for seed in tqdm(SEEDS, desc='Multi-CNN fusion, seeds'):
    sp = SPLITS[seed]
    res, info = multicnn_fusion(sp['train'], sp['train'], sp['val'], sp['test'], seed)
    for name, (yhat, sc) in res.items():
        mf_hold.append({'seed': seed, 'strategy': name, **compute_metrics(LABELS[sp['test']], yhat, sc)})
        store_pred('multicnn_fusion', name, seed, sp['test'], yhat, sc)
    mf_info.append({'protocol': 'holdout', 'run': seed, **{f'val_acc_{c}': v for c, v in info['acc_val'].items()},
                    'best_single': info['best_single'], 'best_subset': info['best_subset'],
                    **{f'w_tau{t:g}_{c}': info['weights'][f'Early fusion, val-weighted (tau={t:g})'][c]
                       for t in CONFIG['fusion_taus'] for c in CNN_NAMES}})
for fi, (tr, te) in enumerate(tqdm(CV_FOLDS, desc='Multi-CNN fusion, CV')):
    in_tr, in_va = group_split(tr, CONFIG['inner_val_frac'], fi)
    res, info = multicnn_fusion(tr, in_tr, in_va, te, MAIN_SEED)
    for name, (yhat, sc) in res.items():
        mf_cv.append({'fold': fi, 'strategy': name, **compute_metrics(LABELS[te], yhat, sc)})
    mf_info.append({'protocol': 'cv', 'run': fi, **{f'val_acc_{c}': v for c, v in info['acc_val'].items()},
                    'best_single': info['best_single'], 'best_subset': info['best_subset'],
                    **{f'w_tau{t:g}_{c}': info['weights'][f'Early fusion, val-weighted (tau={t:g})'][c]
                       for t in CONFIG['fusion_taus'] for c in CNN_NAMES}})
MF_HOLD, MF_CV, MF_INFO = pd.DataFrame(mf_hold), pd.DataFrame(mf_cv), pd.DataFrame(mf_info)
MF_HOLD.to_csv(os.path.join(DIRS['tables'], 'raw_holdout_multicnn_fusion.csv'), index=False)
MF_CV.to_csv(os.path.join(DIRS['tables'], 'raw_cv_multicnn_fusion.csv'), index=False)
MF_INFO.to_csv(os.path.join(DIRS['tables'], 'table_multicnn_fusion_weights_and_selection.csv'), index=False)

In [ ]:
REF = 'Single CNN (val-selected)'
STRATS = list(MF_CV.strategy.unique())
MF_MAT = MF_CV.pivot(index='fold', columns='strategy', values='acc')
rows = []
for s in STRATS:
    cvv = MF_CV[MF_CV.strategy == s]; hov = MF_HOLD[MF_HOLD.strategy == s]
    row = {'strategy': s,
           'cv_acc_mean_%': cvv.acc.mean() * 100, 'cv_acc_sd_%': cvv.acc.std(ddof=1) * 100,
           'cv_mcc_mean': cvv.mcc.mean(), 'holdout_test_acc_mean_%': hov.acc.mean() * 100,
           'holdout_test_acc_sd_%': hov.acc.std(ddof=1) * 100, 'holdout_mcc_mean': hov.mcc.mean()}
    if s != REF:
        diff = MF_MAT[s].values - MF_MAT[REF].values
        r = corrected_ttest(diff)
        row.update({'cv_diff_vs_single_pp': r['mean_diff'] * 100, 'cv_ci_low_pp': r['ci_low'] * 100,
                    'cv_ci_high_pp': r['ci_high'] * 100, 'cv_p': r['p'],
                    'cv_folds_win/tie/loss': f'{(diff > 0).sum()}/{(diff == 0).sum()}/{(diff < 0).sum()}'})
        wins = []
        for seed in SEEDS:
            ref_acc = MF_HOLD[(MF_HOLD.strategy == REF) & (MF_HOLD.seed == seed)].acc.item()
            s_acc = MF_HOLD[(MF_HOLD.strategy == s) & (MF_HOLD.seed == seed)].acc.item()
            wins.append(np.sign(s_acc - ref_acc))
        wins = np.array(wins)
        row['holdout_seeds_win/tie/loss'] = f'{(wins > 0).sum()}/{(wins == 0).sum()}/{(wins < 0).sum()}'
    rows.append(row)
MF_SUM = pd.DataFrame(rows)
mask_ = MF_SUM.strategy != REF
MF_SUM.loc[mask_, 'cv_p_holm'] = holm(MF_SUM.loc[mask_, 'cv_p'].values)
save_table(MF_SUM.round(5), 'table_multicnn_fusion_summary.csv')
print('Multi-CNN fusion strategies (RGB, SVM):')
print(MF_SUM.round(3).to_string(index=False))

mc_rows = []
for s in STRATS:
    if s == REF:
        continue
    for seed in SEEDS:
        a = PRED[('multicnn_fusion', s, seed)]; b = PRED[('multicnn_fusion', REF, seed)]
        mc_rows.append({'strategy_A': s, 'strategy_B': REF, 'seed': seed, **mcnemar_test(a['y_true'], a['y_pred'], b['y_pred'])})
MF_MCN = pd.DataFrame(mc_rows); MF_MCN['p_holm'] = holm(MF_MCN.p_exact)
save_table(MF_MCN, 'table_multicnn_fusion_mcnemar_vs_single.csv')
print('\nMcNemar (fusion strategy vs. val-selected single CNN, each seed; Holm over all tests):')
print(MF_MCN.to_string(index=False))

w_cols = [c for c in MF_INFO.columns if c.startswith('w_tau')]
w_sum = MF_INFO.groupby('protocol')[w_cols + [f'val_acc_{c}' for c in CNN_NAMES]].agg(['mean', 'std']).T.round(4)
save_table(w_sum, 'table_multicnn_fusion_weights_mean_sd.csv', index=True)
print('\nValidation accuracies and fusion weights (mean, SD):')
print(w_sum.to_string())
print('\nSelected single CNN (counts):', MF_INFO.groupby('protocol').best_single.value_counts().to_dict())
print('Selected subset (counts):', MF_INFO.groupby('protocol').best_subset.value_counts().to_dict())

In [ ]:
# tau-sensitivity analysis
taus = CONFIG['fusion_taus']
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
means, los, his = [], [], []
for t in taus:
    m_, lo_, hi_ = corrected_ci(MF_MAT[f'Early fusion, val-weighted (tau={t:g})'].values)
    means.append(m_ * 100); los.append(lo_ * 100); his.append(hi_ * 100)
axes[0].errorbar(range(len(taus)), means, yerr=[np.array(means) - los, np.array(his) - means], fmt='o-', capsize=4,
                 label='Val-weighted early fusion')
for lab, ls in [('Early fusion, equal weights', '--'), (REF, ':'), ('Early fusion, dimension-normalized', '-.')]:
    axes[0].axhline(MF_MAT[lab].mean() * 100, ls=ls, color='k' if lab == REF else 'gray', label=lab)
axes[0].set_xticks(range(len(taus))); axes[0].set_xticklabels([f'{t:g}' for t in taus])
axes[0].set_xlabel('Softmax temperature τ'); axes[0].set_ylabel('CV accuracy (%), mean and corrected 95 % CI')
axes[0].set_title(f'Sensitivity of weighted fusion to τ ({len(CV_FOLDS)} CV folds)'); axes[0].legend(fontsize=8)
wm = MF_INFO[MF_INFO.protocol == 'cv']
for c in CNN_NAMES:
    axes[1].plot(range(len(taus)), [wm[f'w_tau{t:g}_{c}'].mean() for t in taus], 'o-', label=c)
axes[1].axhline(1 / 3, color='gray', ls='--', lw=0.8)
axes[1].set_xticks(range(len(taus))); axes[1].set_xticklabels([f'{t:g}' for t in taus])
axes[1].set_xlabel('Softmax temperature τ'); axes[1].set_ylabel('Mean weight (computed from validation accuracy)')
axes[1].set_title('Computed fusion weights vs. τ'); axes[1].legend()
plt.tight_layout(); save_fig('fig_fusion_tau_sensitivity.png'); plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
mfp = MF_CV.copy(); mfp['acc_%'] = mfp.acc * 100
ordr = MF_CV.groupby('strategy').acc.mean().sort_values(ascending=False).index
sns.boxplot(data=mfp, x='strategy', y='acc_%', order=ordr, ax=ax, color='#9ecae1')
ax.set_xticklabels(ordr, rotation=35, ha='right', fontsize=8); ax.set_xlabel(''); ax.set_ylabel('CV accuracy (%)')
ax.set_title(f'Multi-CNN fusion strategies vs. validation-selected single CNN ({len(CV_FOLDS)} CV folds)')
plt.tight_layout(); save_fig('fig_multicnn_fusion_strategies.png'); plt.show()

## 14. Experiment 5 — SVM hyperparameter sensitivity (selected CNN)
Grid search with inner stratified group 5-fold CV on the seed-42 **training set only**. The grid-selected and
the default (C = 10, γ = 'scale') configurations are then each evaluated once on the seed-42 test set.
The CV accuracies in this section therefore refer to the seed-42 training set, whereas Section 9
uses the larger development set (training + validation).

In [ ]:
X = FEATS[(BEST_CNN, BEST_MOD)]
tr0, te0 = sp0['train'], sp0['test']
grid = {'clf__C': [0.1, 1, 10, 100], 'clf__gamma': ['scale', 'auto', 0.001, 0.01]}
gs = GridSearchCV(make_pipeline('SVM', MAIN_SEED), grid, scoring='accuracy', n_jobs=-1, refit=True,
                  cv=StratifiedGroupKFold(5, shuffle=True, random_state=MAIN_SEED))
gs.fit(X[tr0], LABELS[tr0], groups=GROUPS[tr0])
gsr = pd.DataFrame(gs.cv_results_)
gsr['C'] = gsr['param_clf__C'].astype(float); gsr['gamma'] = gsr['param_clf__gamma'].astype(str)
heat = gsr.pivot(index='gamma', columns='C', values='mean_test_score') * 100
majority = np.bincount(LABELS[tr0]).max() / len(tr0) * 100
default = make_pipeline('SVM', MAIN_SEED).fit(X[tr0], LABELS[tr0])
acc_gs = accuracy_score(LABELS[te0], gs.predict(X[te0])); acc_def = accuracy_score(LABELS[te0], default.predict(X[te0]))
def_cv = gsr[(gsr.C == CONFIG['svm_C']) & (gsr.gamma == str(CONFIG['svm_gamma']))].mean_test_score.item()
gs_tab = gsr[['C', 'gamma', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score')
save_table(gs_tab, 'table_gridsearch_svm.csv')
print(f'Model: {BEST_MOD}+{BEST_CNN}+SVM | training set n = {len(tr0)} | majority-class rate = {majority:.2f} %')
print(f'Grid-selected: {gs.best_params_} | inner-CV acc = {gs.best_score_ * 100:.2f} % | '
      f'test acc = {acc_gs * 100:.2f} % ({int(round(acc_gs * len(te0)))}/{len(te0)})')
print(f'Default (C={CONFIG["svm_C"]}, gamma={CONFIG["svm_gamma"]}): inner-CV acc = {def_cv * 100:.2f} % | '
      f'test acc = {acc_def * 100:.2f} % ({int(round(acc_def * len(te0)))}/{len(te0)})')
degenerate = gsr[np.isclose(gsr.mean_test_score * 100, majority, atol=0.5)]
if len(degenerate):
    print('Configurations at the majority-class rate (degenerate classifiers):',
          degenerate[['C', 'gamma']].values.tolist())

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(heat, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax, cbar_kws={'label': 'Inner 5-fold CV accuracy (%)'})
ax.set_title(f'SVM grid search — {BEST_MOD}+{BEST_CNN} (seed-42 training set)\nmajority-class rate = {majority:.2f} %')
plt.tight_layout(); save_fig('fig_gridsearch_svm.png'); plt.show()

## 15. Learning curve (selected model)
Computed on the development set with stratified group 5-fold CV. Training subsets are drawn **after shuffling**
(`shuffle=True`), starting from 5 % of the fold training data.

In [ ]:
Xd, yd = FEATS[(BEST_CNN, BEST_MOD)][DEV_IDX], LABELS[DEV_IDX]
sizes_lc, tr_sc, va_sc = learning_curve(make_pipeline(BEST_CLF, MAIN_SEED), Xd, yd,
                                        train_sizes=np.array(CONFIG['lc_sizes']),
                                        cv=StratifiedGroupKFold(5, shuffle=True, random_state=MAIN_SEED),
                                        groups=GROUPS[DEV_IDX], scoring='accuracy', shuffle=True,
                                        random_state=MAIN_SEED, n_jobs=-1)
lc_df = pd.DataFrame({'train_size': sizes_lc, 'train_acc_mean_%': tr_sc.mean(1) * 100, 'train_acc_sd_%': tr_sc.std(1, ddof=1) * 100,
                      'val_acc_mean_%': va_sc.mean(1) * 100, 'val_acc_sd_%': va_sc.std(1, ddof=1) * 100})
save_table(lc_df.round(3), 'table_learning_curve.csv')
print(lc_df.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(sizes_lc, lc_df['train_acc_mean_%'], 'o-', label='Training')
ax.fill_between(sizes_lc, lc_df['train_acc_mean_%'] - lc_df['train_acc_sd_%'],
                lc_df['train_acc_mean_%'] + lc_df['train_acc_sd_%'], alpha=0.15)
ax.plot(sizes_lc, lc_df['val_acc_mean_%'], 's-', label='Validation')
ax.fill_between(sizes_lc, lc_df['val_acc_mean_%'] - lc_df['val_acc_sd_%'],
                lc_df['val_acc_mean_%'] + lc_df['val_acc_sd_%'], alpha=0.15)
ax.set_xlabel('Number of training images'); ax.set_ylabel('Accuracy (%)'); ax.legend()
ax.set_title(f'Learning curve — {BEST_KEY} (development set, shuffled subsets, mean ± SD over 5 folds)')
plt.tight_layout(); save_fig('fig_learning_curve.png'); plt.show()

## 16. t-SNE visualization of the feature spaces (RGB)
Unsupervised visualization only (standardization + PCA to 50 dimensions + t-SNE, perplexity 30).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, cnn in zip(axes, CNN_NAMES):
    Z = PCA(n_components=min(50, N - 1), random_state=MAIN_SEED).fit_transform(StandardScaler().fit_transform(FEATS[(cnn, 'RGB')]))
    E = TSNE(n_components=2, perplexity=min(30, (N - 1) // 3), random_state=MAIN_SEED, init='pca').fit_transform(Z)
    for ci in range(2):
        ax.scatter(E[LABELS == ci, 0], E[LABELS == ci, 1], s=8, alpha=0.6, label=CLASS_NAMES[ci])
    ax.set_title(cnn); ax.set_xticks([]); ax.set_yticks([]); ax.legend()
plt.suptitle('t-SNE of deep features (RGB)', fontweight='bold')
plt.tight_layout(); save_fig('fig_tsne.png'); plt.show()

## 17. Experiment 6 — hand-crafted feature baselines and McNemar tests
HOG (9 orientations, 8 × 8 cells, 2 × 2 blocks, L2-Hys) and uniform LBP (P = 24, R = 3, 26-bin
histogram) on 64 × 64 grayscale images, each followed by StandardScaler + SVM (RBF, C = 10,
γ = 'scale'). Same files and same splits as all other experiments. Each baseline is compared with the
selected hybrid model on every seed with the exact McNemar test (2 × 2 tables reported; Holm correction).

In [ ]:
IMG64 = np.stack([rgb2gray(load_resized(p, 64)) for p in tqdm(PATHS, desc='Loading 64x64 images')])
HOG_F = np.stack([hog(g, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm='L2-Hys') for g in IMG64])
P_LBP, R_LBP = 24, 3
LBP_F = np.stack([np.histogram(local_binary_pattern((g * 255).astype(np.uint8), P_LBP, R_LBP, method='uniform').ravel(),
                               bins=np.arange(0, P_LBP + 3), density=True)[0] for g in IMG64])
print(f'HOG dimension: {HOG_F.shape[1]} | LBP dimension: {LBP_F.shape[1]}')

base_rows, mcn_rows = [], []
for seed in SEEDS:
    sp = SPLITS[seed]
    for name, F in [('HOG+SVM', HOG_F), ('LBP+SVM', LBP_F)]:
        pipe = Pipeline([('scaler', StandardScaler()), ('clf', make_classifier('SVM', seed))]).fit(F[sp['train']], LABELS[sp['train']])
        te = sp['test']; yhat = pipe.predict(F[te]); sc = pipe.decision_function(F[te])
        base_rows.append({'seed': seed, 'model': name, **compute_metrics(LABELS[te], yhat, sc)})
        store_pred('baseline', name, seed, te, yhat, sc)
        best_p = PRED[('single', BEST_KEY, seed)]
        mcn_rows.append({'model_A': BEST_KEY, 'model_B': name, 'seed': seed,
                         **mcnemar_test(LABELS[te], best_p['y_pred'], yhat)})
BASE = pd.DataFrame(base_rows); BASE_MCN = pd.DataFrame(mcn_rows)
BASE_MCN['p_holm'] = holm(BASE_MCN.p_exact)
base_tab = pm_table(BASE, ['model'])
save_table(base_tab, 'table_handcrafted_baselines_mean_sd.csv', index=True)
save_table(BASE_MCN, 'table_mcnemar_selected_vs_handcrafted.csv')
print(base_tab.to_string())
print('\nMcNemar 2x2 tables (A = selected hybrid model, B = baseline; exact binomial test):')
print(BASE_MCN.to_string(index=False))

## 18. Experiment 7 — end-to-end fine-tuned CNNs (baseline)
All layers of each ImageNet CNN are fine-tuned with a new linear softmax head (2 classes) on the same
training sets, with training-only augmentation (horizontal / vertical flip, rotation ±15°, mild color
jitter). AdamW (backbone lr 1e-4, head lr 1e-3, weight decay 1e-4), cosine schedule, mixed precision on
GPU, at most 15 epochs, early stopping on validation accuracy (patience 5). The checkpoint with the
highest validation accuracy is evaluated once on the test set. These models are also used for Grad-CAM.

In [ ]:
AUG_TF = T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(15),
                    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1)])
TO_TENSOR = T.Compose([T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


class ArrayImageDataset(Dataset):
    def __init__(self, indices, augment=False, modality='RGB'):
        self.indices, self.augment, self.modality = np.asarray(indices), augment, modality

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = self.indices[i]
        img = Image.fromarray(IMG_RGB[j])
        if self.augment:
            img = AUG_TF(img)
        if self.modality != 'RGB':
            img = img.convert('L').convert('RGB')
        return TO_TENSOR(img), int(LABELS[j])


class FineTuneNet(nn.Module):
    def __init__(self, cnn, pretrained=True):
        super().__init__()
        self.backbone, self.target_layer, dim, _ = build_backbone(cnn, pretrained, to_device=False)
        self.head = nn.Linear(dim, 2)

    def forward(self, x):
        return self.head(self.backbone(x))


USE_AMP = DEVICE.type == 'cuda'
try:
    def make_scaler():
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    make_scaler()
except (AttributeError, TypeError):
    def make_scaler():
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


@torch.no_grad()
def predict_net(net, indices, bs=64):
    net.eval(); probs, loss_sum = [], 0.0
    for s in range(0, len(indices), bs):
        idx = indices[s:s + bs]
        x = arrays_to_tensor(get_images(idx, 'RGB')).to(DEVICE)
        logits = net(x).float()
        loss_sum += nn.functional.cross_entropy(logits, torch.as_tensor(LABELS[idx], device=DEVICE), reduction='sum').item()
        probs.append(torch.softmax(logits, 1)[:, 1].cpu().numpy())
    p = np.concatenate(probs)
    return p, loss_sum / len(indices)


def train_finetune(cnn, seed):
    ckpt = os.path.join(DIRS['checkpoints'], f'finetuned_{cnn}_seed{seed}.pt')
    hist_path = os.path.join(DIRS['tables'], f'finetune_history_{cnn}_seed{seed}.csv')
    set_seed(seed)
    net = FineTuneNet(cnn, CONFIG['pretrained']).to(DEVICE)
    if os.path.exists(ckpt) and not CONFIG['force_retrain']:
        net.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        hist = pd.read_csv(hist_path) if os.path.exists(hist_path) else None
        return net, hist, 'loaded'
    sp = SPLITS[seed]
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(ArrayImageDataset(sp['train'], augment=True), batch_size=CONFIG['ft_batch_size'],
                        shuffle=True, num_workers=0, generator=g, drop_last=False)
    opt = torch.optim.AdamW([{'params': net.backbone.parameters(), 'lr': CONFIG['ft_lr_backbone']},
                             {'params': net.head.parameters(), 'lr': CONFIG['ft_lr_head']}],
                            weight_decay=CONFIG['ft_weight_decay'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG['ft_epochs'])
    scaler = make_scaler(); crit = nn.CrossEntropyLoss()
    best = (-1, np.inf); best_state = None; hist = []; wait = 0
    for ep in range(1, CONFIG['ft_epochs'] + 1):
        net.train(); tl, tc, tn = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out = net(xb); loss = crit(out, yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(yb); tc += (out.argmax(1) == yb).sum().item(); tn += len(yb)
        sched.step()
        pv, vloss = predict_net(net, sp['val'])
        vacc = accuracy_score(LABELS[sp['val']], (pv >= 0.5).astype(int))
        hist.append({'epoch': ep, 'train_loss': tl / tn, 'train_acc': tc / tn, 'val_loss': vloss, 'val_acc': vacc})
        if (vacc, -vloss) > (best[0], -best[1]):
            best = (vacc, vloss); best_state = copy.deepcopy(net.state_dict()); wait = 0; best_ep = ep
        else:
            wait += 1
        print(f'  {cnn} seed {seed} | epoch {ep:2d} | train loss {tl / tn:.4f} acc {tc / tn * 100:.2f} % | '
              f'val loss {vloss:.4f} acc {vacc * 100:.2f} %')
        if wait >= CONFIG['ft_patience']:
            break
    net.load_state_dict(best_state)
    torch.save(best_state, ckpt)
    hist = pd.DataFrame(hist); hist['best_epoch'] = best_ep; hist.to_csv(hist_path, index=False)
    return net, hist, f'trained (best epoch {best_ep})'


FT_NETS, ft_rows, ft_mcn = {}, [], []
if CONFIG['run_finetune']:
    for cnn in CONFIG['ft_models']:
        for seed in FT_SEEDS:
            t0 = time.perf_counter()
            net, hist, status = train_finetune(cnn, seed)
            te = SPLITS[seed]['test']
            p, _ = predict_net(net, te); yhat = (p >= 0.5).astype(int)
            name = f'Fine-tuned {cnn} (end-to-end)'
            ft_rows.append({'seed': seed, 'model': name, 'cnn': cnn, 'status': status,
                            'time_min': (time.perf_counter() - t0) / 60, **compute_metrics(LABELS[te], yhat, p)})
            store_pred('finetune', name, seed, te, yhat, p)
            ft_mcn.append({'model_A': BEST_KEY, 'model_B': name, 'seed': seed,
                           **mcnemar_test(LABELS[te], PRED[('single', BEST_KEY, seed)]['y_pred'], yhat)})
            print(f'{name} | seed {seed} | {status} | test acc {ft_rows[-1]["acc"] * 100:.2f} %')
            if seed == MAIN_SEED:
                FT_NETS[cnn] = net.eval()
            else:
                del net
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
    FT = pd.DataFrame(ft_rows); FT_MCN = pd.DataFrame(ft_mcn); FT_MCN['p_holm'] = holm(FT_MCN.p_exact)
    FT.to_csv(os.path.join(DIRS['tables'], 'raw_holdout_finetuned_cnns.csv'), index=False)
    ft_tab = pm_table(FT, ['model'])
    save_table(ft_tab, 'table_finetuned_cnns_mean_sd.csv', index=True)
    save_table(FT_MCN, 'table_mcnemar_selected_vs_finetuned.csv')
    print(f'\nEnd-to-end fine-tuned CNNs, test set (mean ± SD over {len(FT_SEEDS)} seeds):')
    print(ft_tab.to_string())
    print('\nMcNemar (A = selected hybrid, B = fine-tuned CNN):')
    print(FT_MCN.to_string(index=False))
    h42 = [pd.read_csv(os.path.join(DIRS['tables'], f'finetune_history_{c}_seed{MAIN_SEED}.csv'))
           for c in CONFIG['ft_models'] if os.path.exists(os.path.join(DIRS['tables'], f'finetune_history_{c}_seed{MAIN_SEED}.csv'))]
    if h42:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        for c, hh in zip(CONFIG['ft_models'], h42):
            axes[0].plot(hh.epoch, hh.train_loss, '-', label=f'{c} train'); axes[0].plot(hh.epoch, hh.val_loss, '--', label=f'{c} val')
            axes[1].plot(hh.epoch, hh.val_acc * 100, 'o-', label=c)
        axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy loss'); axes[0].legend(fontsize=8)
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation accuracy (%)'); axes[1].legend()
        plt.suptitle(f'Fine-tuning curves (seed {MAIN_SEED})'); plt.tight_layout(); save_fig('fig_finetune_curves.png'); plt.show()
else:
    FT = pd.DataFrame(); FT_MCN = pd.DataFrame()
    print('Fine-tuning disabled in CONFIG.')

## 19. Experiment 8 — training-data augmentation for the selected hybrid model
For each seed, `aug_copies` augmented versions of every *training* image (same augmentation as in
Section 18) are passed through the frozen backbone and added to the training features. Validation and test
images are not augmented. Results are compared with the non-augmented model on the same splits.

In [ ]:
AUG = pd.DataFrame()
if CONFIG['run_augmentation']:
    model_b, _ = BACKBONES[BEST_CNN]
    aug_rows = []
    for seed in tqdm(SEEDS, desc='Augmentation'):
        set_seed(seed); sp = SPLITS[seed]; tr = sp['train']
        feats_aug, ys_aug = [], []
        with torch.no_grad():
            for c in range(CONFIG['aug_copies']):
                for s in range(0, len(tr), CONFIG['batch_size']):
                    idx = tr[s:s + CONFIG['batch_size']]
                    imgs = []
                    for j in idx:
                        im = AUG_TF(Image.fromarray(IMG_RGB[j]))
                        if BEST_MOD != 'RGB':
                            im = im.convert('L').convert('RGB')
                        imgs.append(np.asarray(im))
                    feats_aug.append(model_b(arrays_to_tensor(np.stack(imgs)).to(DEVICE)).float().cpu().numpy())
                    ys_aug.append(LABELS[idx])
        X = FEATS[(BEST_CNN, BEST_MOD)]
        Xtr = np.vstack([X[tr]] + feats_aug); ytr = np.concatenate([LABELS[tr]] + ys_aug)
        pipe = make_pipeline(BEST_CLF, seed).fit(Xtr, ytr)
        for part in ['val', 'test']:
            idx = sp[part]; yhat = pipe.predict(X[idx]); sc = model_scores(pipe, X[idx])
            aug_rows.append({'seed': seed, 'model': f'{BEST_KEY} + augmentation (x{CONFIG["aug_copies"] + 1} train)',
                             'part': part, 'n_train': len(ytr), **compute_metrics(LABELS[idx], yhat, sc)})
            if part == 'test':
                store_pred('augmentation', BEST_KEY + ' + aug', seed, idx, yhat, sc)
    AUG = pd.DataFrame(aug_rows)
    base_t = HOLD[(HOLD.model == BEST_KEY) & (HOLD.part == 'test')].set_index('seed').acc
    aug_t = AUG[AUG.part == 'test'].set_index('seed').acc
    d = (aug_t - base_t.loc[aug_t.index]) * 100
    comp = pd.DataFrame({'no_aug_test_acc_%': base_t * 100, 'aug_test_acc_%': aug_t * 100, 'diff_pp': d})
    save_table(comp.round(3), 'table_augmentation_selected_model.csv', index=True)
    print(comp.round(2).to_string())
    print(f'Mean difference (augmented − original): {d.mean():.2f} ± {d.std(ddof=1):.2f} pp; '
          f'paired t-test p = {stats.ttest_rel(aug_t, base_t.loc[aug_t.index]).pvalue:.3f}')
else:
    print('Augmentation experiment disabled in CONFIG.')

## 20. Explainability

* **Occlusion sensitivity (deployed pipeline).** The complete hybrid pipeline (frozen CNN → scaler → PCA →
  classifier) of the selected model, fitted on the seed-42 training set, is explained on **all** seed-42
  test images. A square patch (32 × 32 px, stride 16) is replaced by the ImageNet mean color; the map value is
  the decrease in the signed SVM decision score toward the true class, averaged over all patches that cover
  a pixel. Positive values mark regions that support the correct class.
* **Grad-CAM (fine-tuned CNN only).** Grad-CAM requires the gradient of a class score. It is therefore applied
  only to the end-to-end fine-tuned CNN of the same architecture (seed 42; last convolutional block; target =
  predicted-class logit). These maps explain the fine-tuned CNN, **not** the hybrid SVM pipeline.
* **Quantification.** For every test image: fraction of positive attribution inside the pistachio mask, the
  mask area fraction (expected value for a spatially uniform map) and whether the maximum lies inside the mask.
* **Example selection.** Shown examples are drawn at random (fixed seed) from the correctly classified and
  from the misclassified test images.

In [ ]:
def true_class_score(pipe, feats, y_true):
    s = model_scores(pipe, feats)
    clf = pipe.steps[-1][1]
    if isinstance(clf, SVC) and not clf.probability:
        return s if y_true == 1 else -s
    return s if y_true == 1 else 1 - s


@torch.no_grad()
def occlusion_map(backbone, pipe, img_uint8, y_true, patch, stride, bs=64):
    x = arrays_to_tensor(img_uint8[None])
    H, W = x.shape[2:]
    pos = [(r, c) for r in range(0, H - patch + 1, stride) for c in range(0, W - patch + 1, stride)]
    s0 = true_class_score(pipe, backbone(x.to(DEVICE)).float().cpu().numpy(), y_true)[0]
    scores = []
    for s in range(0, len(pos), bs):
        chunk = pos[s:s + bs]
        xb = x.repeat(len(chunk), 1, 1, 1)
        for k, (r, c) in enumerate(chunk):
            xb[k, :, r:r + patch, c:c + patch] = 0.0      # 0 after normalization = ImageNet mean color
        scores.extend(true_class_score(pipe, backbone(xb.to(DEVICE)).float().cpu().numpy(), y_true))
    heat = np.zeros((H, W)); cnt = np.zeros((H, W))
    for (r, c), sc in zip(pos, scores):
        heat[r:r + patch, c:c + patch] += s0 - sc; cnt[r:r + patch, c:c + patch] += 1
    return heat / np.maximum(cnt, 1), s0


def attribution_stats(heat, mask):
    pos = np.clip(heat, 0, None); tot = pos.sum()
    r, c = np.unravel_index(np.argmax(heat), heat.shape)
    return {'frac_inside_mask': (pos * mask).sum() / tot if tot > 0 else np.nan,
            'mask_area_frac': mask.mean(), 'max_inside_mask': bool(mask[r, c])}


te0 = sp0['test']
pipe_x = make_pipeline(BEST_CLF, MAIN_SEED).fit(FEATS[(BEST_CNN, BEST_MOD)][sp0['train']], LABELS[sp0['train']])
bb_x, _ = BACKBONES[BEST_CNN]
yhat_x = PRED[('single', BEST_KEY, MAIN_SEED)]['y_pred']
OCC, occ_rows = {}, []
for k, j in enumerate(tqdm(te0, desc='Occlusion (full test set)')):
    heat, s0 = occlusion_map(bb_x, pipe_x, get_images([j], BEST_MOD)[0], LABELS[j],
                             CONFIG['occlusion_patch'], CONFIG['occlusion_stride'])
    OCC[j] = heat
    occ_rows.append({'index': j, 'y_true': LABELS[j], 'y_pred': yhat_x[k], 'correct': yhat_x[k] == LABELS[j],
                     'base_score_true_class': s0, **attribution_stats(heat, MASKS[j])})
OCC_DF = pd.DataFrame(occ_rows)
OCC_DF.to_csv(os.path.join(DIRS['predictions'], 'occlusion_attribution_per_image.csv'), index=False)

In [ ]:
GC, gc_rows = {}, []
if CONFIG['run_finetune'] and BEST_CNN in FT_NETS:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    net = FT_NETS[BEST_CNN].eval()
    for prm in net.parameters():
        prm.requires_grad_(True)
    ft_pred = PRED[('finetune', f'Fine-tuned {BEST_CNN} (end-to-end)', MAIN_SEED)]['y_pred']
    cam = GradCAM(model=net, target_layers=[net.target_layer])
    for s in tqdm(range(0, len(te0), 16), desc='Grad-CAM (fine-tuned CNN)'):
        idx = te0[s:s + 16]
        x = arrays_to_tensor(get_images(idx, 'RGB')).to(DEVICE)
        maps = cam(input_tensor=x, targets=[ClassifierOutputTarget(int(c)) for c in ft_pred[s:s + 16]])
        for k, j in enumerate(idx):
            GC[j] = maps[k]
            gc_rows.append({'index': j, 'y_true': LABELS[j], 'y_pred_finetuned': ft_pred[s + k],
                            'correct': ft_pred[s + k] == LABELS[j], **attribution_stats(maps[k], MASKS[j])})
GC_DF = pd.DataFrame(gc_rows)
if len(GC_DF):
    GC_DF.to_csv(os.path.join(DIRS['predictions'], 'gradcam_attribution_per_image_finetuned.csv'), index=False)

In [ ]:
xai_rows = []
for name, df in [(f'Occlusion — {BEST_KEY} (deployed hybrid pipeline)', OCC_DF),
                 (f'Grad-CAM — fine-tuned {BEST_CNN} (end-to-end)', GC_DF)]:
    if not len(df):
        continue
    for grp, sub in [('all', df), ('correct', df[df.correct]), ('misclassified', df[~df.correct])]:
        if not len(sub):
            continue
        row = {'method': name, 'group': grp, 'n': len(sub),
               'frac_inside_mask_mean': sub.frac_inside_mask.mean(), 'frac_inside_mask_sd': sub.frac_inside_mask.std(ddof=1),
               'mask_area_frac_mean': sub.mask_area_frac.mean(), 'pointing_game_%': sub.max_inside_mask.mean() * 100}
        if len(sub) > 5:
            d = (sub.frac_inside_mask - sub.mask_area_frac).dropna()
            row['wilcoxon_p_(inside>area)'] = stats.wilcoxon(d, alternative='greater').pvalue if (d != 0).any() else np.nan
        xai_rows.append(row)
XAI = pd.DataFrame(xai_rows)
save_table(XAI.round(4), 'table_xai_quantitative.csv')
print('Quantitative attribution analysis (seed-42 test set):')
print(XAI.round(3).to_string(index=False))

rng = np.random.default_rng(MAIN_SEED)
cor = te0[yhat_x == LABELS[te0]]; mis = te0[yhat_x != LABELS[te0]]
ex = list(rng.choice(cor, min(CONFIG['n_examples'], len(cor)), replace=False))
ex_mis = list(rng.choice(mis, min(CONFIG['n_examples'], len(mis)), replace=False)) if len(mis) else []
show_idx = [(j, 'correct') for j in ex] + [(j, 'misclassified') for j in ex_mis]
ncol = 4 if len(GC) else 3
fig, axes = plt.subplots(len(show_idx), ncol, figsize=(3.4 * ncol, 3.2 * len(show_idx)))
axes = np.atleast_2d(axes)
pos_of = {j: k for k, j in enumerate(te0)}
for r, (j, grp) in enumerate(show_idx):
    img = IMG_RGB[j]
    axes[r, 0].imshow(img); axes[r, 0].contour(MASKS[j], colors='lime', linewidths=0.8)
    axes[r, 0].set_title(f'#{j} {grp}\ntrue {CLASS_NAMES[LABELS[j]]}, hybrid pred {CLASS_NAMES[yhat_x[pos_of[j]]]}', fontsize=8)
    h_ = OCC[j]; hp = np.clip(h_, 0, None) / (np.abs(h_).max() + 1e-12)
    axes[r, 1].imshow(h_, cmap='RdBu_r', vmin=-np.abs(h_).max(), vmax=np.abs(h_).max())
    axes[r, 1].set_title('Occlusion map (hybrid)\nred = supports true class', fontsize=8)
    axes[r, 2].imshow(img); axes[r, 2].imshow(hp, cmap='jet', alpha=0.45)
    axes[r, 2].set_title(f'Occlusion overlay\ninside mask: {OCC_DF.set_index("index").loc[j, "frac_inside_mask"] * 100:.0f} %', fontsize=8)
    if len(GC):
        axes[r, 3].imshow(img); axes[r, 3].imshow(GC[j], cmap='jet', alpha=0.45)
        gp = GC_DF.set_index('index').loc[j]
        axes[r, 3].set_title(f'Grad-CAM (fine-tuned {BEST_CNN})\npred {CLASS_NAMES[int(gp.y_pred_finetuned)]}, '
                             f'inside mask: {gp.frac_inside_mask * 100:.0f} %', fontsize=8)
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
fig.suptitle('Randomly selected test examples (fixed seed): correctly classified and misclassified', fontweight='bold', y=1.005)
plt.tight_layout(); save_fig('fig_xai_examples.png'); plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
data_ = [OCC_DF.frac_inside_mask.dropna() * 100]; labs_ = ['Occlusion\n(hybrid pipeline)']
if len(GC_DF):
    data_.append(GC_DF.frac_inside_mask.dropna() * 100); labs_.append(f'Grad-CAM\n(fine-tuned {BEST_CNN})')
ax.boxplot(data_)
ax.set_xticks(range(1, len(labs_) + 1)); ax.set_xticklabels(labs_)
ax.axhline(OCC_DF.mask_area_frac.mean() * 100, color='r', ls='--', label='Mask area (uniform map)')
ax.set_ylabel('Positive attribution inside pistachio mask (%)'); ax.legend()
ax.set_title(f'Attribution inside the pistachio region — all {len(te0)} test images (seed {MAIN_SEED})')
plt.tight_layout(); save_fig('fig_xai_quantitative.png'); plt.show()

### 20.1 SHAP on the classifier input (PCA components)
KernelExplainer (model-agnostic; suitable for an RBF-SVM) explaining the SVM decision function of the
selected model with respect to its inputs, i.e. the PCA components of the standardized deep features.
Background: k-means summary (50 centroids) of the seed-42 training set; 100 randomly selected test images;
500 coalition samples per explanation. **PCA components are linear combinations of deep features and have
no direct semantic, morphological or biological meaning**; this analysis only shows how the classifier
uses the reduced feature space.

In [ ]:
if CONFIG['run_shap']:
    import shap
    Xf = FEATS[(BEST_CNN, BEST_MOD)]
    pre = Pipeline(pipe_x.steps[:-1])
    Ztr = pre.transform(Xf[sp0['train']]); Zte = pre.transform(Xf[te0])
    clf_x = pipe_x.steps[-1][1]
    f_shap = (lambda Z: clf_x.decision_function(Z)) if isinstance(clf_x, SVC) else (lambda Z: clf_x.predict_proba(Z)[:, 1])
    bg = shap.kmeans(Ztr, min(CONFIG['shap_background'], len(Ztr)))
    expl = shap.KernelExplainer(f_shap, bg)
    rng = np.random.default_rng(MAIN_SEED)
    sel = rng.choice(len(Zte), min(CONFIG['shap_n_explain'], len(Zte)), replace=False)
    sv = np.asarray(expl.shap_values(Zte[sel], nsamples=CONFIG['shap_nsamples']))
    if sv.ndim == 3:
        sv = sv[..., 0]
    names = [f'PC{i + 1}' for i in range(Zte.shape[1])]
    imp = pd.DataFrame({'component': names, 'mean_abs_shap': np.abs(sv).mean(0),
                        'explained_variance_%': pipe_x.named_steps['pca'].explained_variance_ratio_ * 100})
    imp = imp.sort_values('mean_abs_shap', ascending=False)
    save_table(imp, 'table_shap_pca_components.csv')
    top = imp.head(10)
    print(f'SHAP: KernelExplainer, background = k-means({CONFIG["shap_background"]}) of training set, '
          f'{len(sel)} test images, nsamples = {CONFIG["shap_nsamples"]}, output = SVM decision function')
    print(f'Top-10 components explain {top["explained_variance_%"].sum():.2f} % of feature variance and '
          f'{top.mean_abs_shap.sum() / imp.mean_abs_shap.sum() * 100:.1f} % of total mean |SHAP|.')
    print(imp.head(20).round(4).to_string(index=False))
    plt.figure()
    shap.summary_plot(sv, Zte[sel], feature_names=names, max_display=15, show=False)
    plt.title(f'SHAP values of PCA components — {BEST_KEY}\n(positive = toward Siirt)', fontsize=10)
    save_fig('fig_shap_pca_components.png'); plt.show()
else:
    print('SHAP disabled in CONFIG.')

## 21. Master results table, per-image predictions and summary

In [ ]:
master = []
for key, mod, cnn, clf in SINGLE_MODELS:
    s = HOLD[(HOLD.model == key) & (HOLD.part == 'test')]; c = CV[CV.model == key]
    master.append({'group': '1. Single CNN hybrid', 'model': key, 'holdout_runs': len(s), 'cv_folds': len(c),
                   'holdout_acc_mean': s.acc.mean(), 'holdout_acc_sd': s.acc.std(ddof=1),
                   'holdout_f1_macro_mean': s.f1_macro.mean(), 'holdout_mcc_mean': s.mcc.mean(),
                   'holdout_kappa_mean': s.kappa.mean(), 'holdout_auc_mean': s.auc.mean(),
                   'cv_acc_mean': c.acc.mean(), 'cv_acc_sd': c.acc.std(ddof=1)})
for key, cnn, clf in RG_MODELS:
    s = RG_HOLD[(RG_HOLD.model == key) & (RG_HOLD.part == 'test')]; c = RG_CV[RG_CV.model == key]
    master.append({'group': '2. RGB+grayscale fusion', 'model': key, 'holdout_runs': len(s), 'cv_folds': len(c),
                   'holdout_acc_mean': s.acc.mean(), 'holdout_acc_sd': s.acc.std(ddof=1),
                   'holdout_f1_macro_mean': s.f1_macro.mean(), 'holdout_mcc_mean': s.mcc.mean(),
                   'holdout_kappa_mean': s.kappa.mean(), 'holdout_auc_mean': s.auc.mean(),
                   'cv_acc_mean': c.acc.mean(), 'cv_acc_sd': c.acc.std(ddof=1)})
for st in STRATS:
    s = MF_HOLD[MF_HOLD.strategy == st]; c = MF_CV[MF_CV.strategy == st]
    master.append({'group': '3. Multi-CNN fusion (RGB, SVM)', 'model': st, 'holdout_runs': len(s), 'cv_folds': len(c),
                   'holdout_acc_mean': s.acc.mean(), 'holdout_acc_sd': s.acc.std(ddof=1),
                   'holdout_f1_macro_mean': s.f1_macro.mean(), 'holdout_mcc_mean': s.mcc.mean(),
                   'holdout_kappa_mean': s.kappa.mean(), 'holdout_auc_mean': s.auc.mean(),
                   'cv_acc_mean': c.acc.mean(), 'cv_acc_sd': c.acc.std(ddof=1)})
for grp, df_ in [('4. Hand-crafted baseline', BASE), ('5. Fine-tuned CNN', FT),
                 ('6. Augmentation', AUG[AUG.part == 'test'] if len(AUG) else AUG)]:
    if not len(df_):
        continue
    for key in df_.model.unique():
        s = df_[df_.model == key]
        master.append({'group': grp, 'model': key, 'holdout_runs': len(s), 'cv_folds': 0,
                       'holdout_acc_mean': s.acc.mean(), 'holdout_acc_sd': s.acc.std(ddof=1),
                       'holdout_f1_macro_mean': s.f1_macro.mean(), 'holdout_mcc_mean': s.mcc.mean(),
                       'holdout_kappa_mean': s.kappa.mean(), 'holdout_auc_mean': s.auc.mean(),
                       'cv_acc_mean': np.nan, 'cv_acc_sd': np.nan})
MASTER = pd.DataFrame(master)
save_table(MASTER.round(6), 'table_MASTER_all_experiments.csv')
pd.set_option('display.width', 250)
disp = MASTER.copy()
for c in ['holdout_acc_mean', 'holdout_acc_sd', 'holdout_f1_macro_mean', 'cv_acc_mean', 'cv_acc_sd']:
    disp[c] = (disp[c] * 100).round(2)
print(disp.round(4).to_string(index=False))

pred_frames = []
for (exp, model, seed), p in PRED.items():
    pred_frames.append(pd.DataFrame({'experiment': exp, 'model': model, 'seed': seed, 'index': p['idx'],
                                     'file': file_df['file'].values[p['idx']], 'y_true': p['y_true'],
                                     'y_pred': p['y_pred'], 'score': p['score']}))
PRED_DF = pd.concat(pred_frames, ignore_index=True)
PRED_DF.to_csv(os.path.join(DIRS['predictions'], 'per_image_test_predictions_all_experiments.csv'), index=False)
print(f'\nSaved {len(PRED_DF)} per-image test predictions ({PRED_DF.groupby(["experiment", "model", "seed"]).ngroups} runs).')

In [ ]:
best_cv = CV_SUM.iloc[0]
summary = {
    'n_images': int(N), 'class_counts': {CLASS_NAMES[i]: int((LABELS == i).sum()) for i in range(2)},
    'split_sizes_seed42': {k: int(len(v)) for k, v in sp0.items()}, 'split_sizes_all_seeds': size_df.to_dict('records'),
    'seeds': SEEDS, 'near_duplicate_grouping': DUP_INFO,
    'cv': f'{CONFIG["cv_repeats"]}x{CONFIG["cv_splits"]} repeated stratified CV on development set (n={len(DEV_IDX)})',
    'selected_model': BEST_KEY,
    'selected_cv_acc': f'{best_cv.cv_acc_mean * 100:.2f} ± {best_cv.cv_acc_sd * 100:.2f} % '
                       f'(95% CI {best_cv.cv_acc_ci_low * 100:.2f}–{best_cv.cv_acc_ci_high * 100:.2f})',
    'selected_cv_f1_macro': f'{best_cv.cv_f1_macro_mean * 100:.2f} ± {best_cv.cv_f1_macro_sd * 100:.2f} %',
    'selected_cv_mcc': f'{best_cv.cv_mcc_mean:.4f} ± {best_cv.cv_mcc_sd:.4f}',
    'selected_holdout_acc_5seeds': f'{bh.acc.mean() * 100:.2f} ± {bh.acc.std(ddof=1) * 100:.2f} %',
    'selected_holdout_mcc_5seeds': f'{bh.mcc.mean():.4f} ± {bh.mcc.std(ddof=1):.4f}',
    'selected_seed42_test': f'{best_m["n_correct"]}/{best_m["n"]} = {best_m["acc"] * 100:.2f} % '
                            f'(bootstrap 95% CI {best_ci["acc"][0] * 100:.2f}–{best_ci["acc"][1] * 100:.2f})',
    'top5_cv': [f'{r.model}: {r.cv_acc_mean * 100:.2f} ± {r.cv_acc_sd * 100:.2f}' for r in CV_SUM.head(5).itertuples()],
    'svm_vs_knn_significant_after_holm': fam_clf[fam_clf.model_B.str.endswith('KNN') & fam_clf.model_A.str.endswith('SVM')][
        ['model_A', 'diff_pp', 'p_holm']].round(4).to_dict('records'),
    'multicnn_fusion': MF_SUM[['strategy', 'cv_acc_mean_%', 'cv_diff_vs_single_pp', 'cv_p_holm',
                               'holdout_test_acc_mean_%', 'holdout_seeds_win/tie/loss']].round(4).to_dict('records'),
    'baselines_mcnemar': BASE_MCN[['model_B', 'seed', 'b_A_correct_B_wrong', 'c_A_wrong_B_correct', 'p_exact', 'p_holm']].to_dict('records'),
    'finetuned': (FT.groupby('model').acc.agg(['mean', 'std']) * 100).round(2).to_dict('index') if len(FT) else None,
    'xai': XAI.round(4).to_dict('records'),
    'near_duplicates': dup_df.round(3).to_dict('records'),
}
with open(os.path.join(OUT, 'summary_for_manuscript.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=str)
print(json.dumps(summary, indent=2, ensure_ascii=False, default=str))
print(f'\nAll outputs saved under: {os.path.abspath(OUT)}')